In [ ]:
# ============================================================
# CELL 1 - PROJECT OVERVIEW
# ============================================================

print("""
This project builds a production-style RAG API for Indian tax FAQs.

FastAPI exposes a POST /chat endpoint with Server-Sent Events (SSE)
streaming, while ChromaDB stores and retrieves relevant knowledge.

The API also includes exact-match caching, JSONL request logging,
latency/token tracking, error handling, and DeepEval evaluation.
""")


This project builds a production-style RAG API for Indian tax FAQs.

FastAPI exposes a POST /chat endpoint with Server-Sent Events (SSE)
streaming, while ChromaDB stores and retrieves relevant knowledge.

The API also includes exact-match caching, JSONL request logging,
latency/token tracking, error handling, and DeepEval evaluation.



In [ ]:
# ============================================================
# CELL 2 - INSTALL DEPENDENCIES
# ============================================================

!pip -q install fastapi uvicorn chromadb requests pydantic \
    deepeval pytest httpx nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 48.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
# ============================================================
# CELL 3 - INSTALL OLLAMA
# ============================================================

!curl -fsSL https://ollama.com/install.sh | sh

print("Ollama installation completed.")

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd
Ollama installation completed.


In [ ]:
# ============================================================
# CELL 3 - INSTALL OLLAMA
# ============================================================

import subprocess

print("Installing zstd dependency...")

result = subprocess.run(
    ["apt-get", "update", "-qq"],
    capture_output=True,
    text=True
)

result = subprocess.run(
    ["apt-get", "install", "-y", "zstd"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Failed to install zstd.")

print("zstd installed successfully.")

print("\nInstalling Ollama...")

result = subprocess.run(
    ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Ollama installation failed.")

print("\nOllama installed successfully.")

Installing zstd dependency...
zstd installed successfully.

Installing Ollama...


Ollama installed successfully.


In [ ]:
# ============================================================
# CELL 4 - VERIFY OLLAMA INSTALLATION
# ============================================================

import subprocess

result = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("Ollama installation could not be verified.")
    print(result.stderr)
else:
    print("Ollama is installed successfully.")
    print("Version:", result.stdout.strip())

Ollama is installed successfully.
Version: Warning: could not connect to a running Ollama instance


In [ ]:
# ============================================================
# CELL 5 - START OLLAMA SERVER
# ============================================================

import subprocess
import time
import requests

print("Starting Ollama server...")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    response = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=10
    )

    if response.status_code == 200:
        print("Ollama server is running successfully.")
        print("Server URL: http://127.0.0.1:11434")
    else:
        print("Ollama server responded with status:", response.status_code)

except Exception as e:
    print("Could not connect to Ollama server.")
    print("Error:", e)

Starting Ollama server...
Ollama server is running successfully.
Server URL: http://127.0.0.1:11434


In [ ]:
# ============================================================
# CELL 6 - DOWNLOAD LOCAL LLM
# ============================================================

print("Downloading Qwen2.5 1.5B model...")
print("This may take a few minutes depending on the connection.\n")

!ollama pull qwen2.5:1.5b

print("\nModel download completed.")

This may take a few minutes depending on the connection.



Model download completed.


In [ ]:
# ============================================================
# CELL 7 - TEST THE LOCAL LLM
# ============================================================

import requests

payload = {
    "model": "qwen2.5:1.5b",
    "prompt": "What is Section 80C of the Indian Income Tax Act?",
    "stream": False
}

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json=payload,
    timeout=120
)

if response.status_code == 200:
    result = response.json()

    print("LLM is working successfully.")
    print("\nModel response:")
    print("-" * 70)
    print(result["response"])
else:
    print("LLM request failed.")
    print("Status code:", response.status_code)
    print(response.text)

LLM is working successfully.

Model response:
----------------------------------------------------------------------
Section 80C of the Indian Income Tax Act, 1961, is a tax relief provision designed to provide tax benefits to individuals who are involved in the production and cultivation of medicinal and aromatic crops. This includes crops such as turmeric, cardamom, pepper, cinnamon, and others. The purpose of this provision is to incentivize the growth of these crops and to encourage farmers to cultivate them, as they are considered essential for the country's healthcare system.

The benefits of Section 80C include the following:

1. It allows for a deduction of 50% of the cost of production and cultivation of medicinal and aromatic crops from the taxable income.

2. The deduction is available for a period of 10 years, after which the deduction ceases.

3. The cost of production includes expenses related to the harvesting and processing of the crops.

4. The crops must be grown on a

In [ ]:
# ============================================================
# CELL 8 - CREATE TAX KNOWLEDGE BASE
# ============================================================

tax_documents = [
    {
        "id": "tax_001",
        "title": "Section 80C",
        "text": """
Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year.
"""
    },
    {
        "id": "tax_002",
        "title": "Form 16",
        "text": """
Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return.
"""
    },
    {
        "id": "tax_003",
        "title": "HRA",
        "text": """
House Rent Allowance, commonly called HRA, is an allowance that may be
provided by an employer as part of salary. Eligible salaried taxpayers
may claim an exemption related to HRA subject to applicable conditions.
"""
    },
    {
        "id": "tax_004",
        "title": "Tax Deduction",
        "text": """
A tax deduction reduces the amount of income considered for taxation.
Eligible deductions are generally claimed from gross total income while
calculating taxable income, subject to the applicable rules.
"""
    },
    {
        "id": "tax_005",
        "title": "Tax Exemption",
        "text": """
A tax exemption generally means that specified income or a specified
portion of income is not included in taxable income when the applicable
conditions are satisfied.
"""
    },
    {
        "id": "tax_006",
        "title": "TDS",
        "text": """
Tax Deducted at Source, or TDS, is a mechanism under which tax is
deducted at the time of specified payments and deposited with the
government on behalf of the recipient.
"""
    },
    {
        "id": "tax_007",
        "title": "Income Tax Return",
        "text": """
An Income Tax Return, or ITR, is a return filed by a taxpayer to report
income, deductions, taxes paid and other required information for the
relevant assessment year.
"""
    },
    {
        "id": "tax_008",
        "title": "Capital Gains",
        "text": """
Capital gains generally arise when a taxpayer transfers a capital asset
for consideration and realizes a gain or loss. The applicable tax
treatment depends on the type of asset, holding period and relevant
tax rules.
"""
    },
    {
        "id": "tax_009",
        "title": "Advance Tax",
        "text": """
Advance tax is income tax paid during the financial year in installments
when the taxpayer meets the conditions requiring payment of advance tax.
The applicable installments and rules depend on the taxpayer's situation.
"""
    },
    {
        "id": "tax_010",
        "title": "New Tax Regime",
        "text": """
India provides different income tax regimes. The applicable deductions,
exemptions and tax rates can differ between regimes and may change
between financial years. Taxpayers should verify the rules applicable
to the relevant year.
"""
    },
    {
        "id": "tax_011",
        "title": "Old Tax Regime",
        "text": """
The old tax regime allows various deductions and exemptions subject to
eligibility conditions. The actual tax treatment depends on the
financial year and taxpayer circumstances.
"""
    },
    {
        "id": "tax_012",
        "title": "GST",
        "text": """
Goods and Services Tax, or GST, is an indirect tax applied to the supply
of goods and services in India. GST is separate from personal income tax.
"""
    }
]

print("Knowledge base created successfully.")
print("Total documents:", len(tax_documents))

print("\nAvailable topics:")
for document in tax_documents:
    print("-", document["title"])

Knowledge base created successfully.
Total documents: 12

Available topics:
- Section 80C
- Form 16
- HRA
- Tax Deduction
- Tax Exemption
- TDS
- Income Tax Return
- Capital Gains
- Advance Tax
- New Tax Regime
- Old Tax Regime
- GST


In [ ]:
# ============================================================
# CELL 9 - CREATE CHROMADB COLLECTION
# ============================================================

import chromadb

# Create a persistent ChromaDB database
chroma_client = chromadb.PersistentClient(path="./chroma_data")

# Create or reuse the collection
collection = chroma_client.get_or_create_collection(
    name="indian_tax_knowledge"
)

print("ChromaDB collection created successfully.")
print("Collection name:", collection.name)

ChromaDB collection created successfully.
Collection name: indian_tax_knowledge


In [ ]:
# ============================================================
# CELL 10 - INSERT DOCUMENTS INTO CHROMADB
# ============================================================

collection.upsert(
    ids=[document["id"] for document in tax_documents],
    documents=[document["text"] for document in tax_documents],
    metadatas=[{"title": document["title"]} for document in tax_documents]
)

print("Documents inserted successfully.")
print("Total documents in ChromaDB:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 38.4MiB/s]


Documents inserted successfully.
Total documents in ChromaDB: 12


In [ ]:
# ============================================================
# CELL 11 - TEST CHROMADB RETRIEVAL
# ============================================================

query = "What is Form 16?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("User Query:")
print(query)

print("\nRetrieved Context:")
print("=" * 70)

for i, document in enumerate(results["documents"][0], start=1):
    print(f"\nContext {i}:")
    print(document.strip())

print("\n" + "=" * 70)
print("Retrieval test completed successfully.")

User Query:
What is Form 16?

Retrieved Context:

Context 1:
Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return.

Context 2:
Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year.

Context 3:
The old tax regime allows various deductions and exemptions subject to
eligibility conditions. The actual tax treatment depends on the
financial year and taxpayer circumstances.

Retrieval test completed successfully.


In [ ]:
# ============================================================
# CELL 12 - CREATE FASTAPI RAG APPLICATION
# ============================================================

%%writefile app.py

import os
import json
import time
import threading
from datetime import datetime, timezone
from typing import Generator

import requests
import chromadb

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = os.getenv("MODEL_NAME", "qwen2.5:1.5b")
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://127.0.0.1:11434"
)

LOG_FILE = os.getenv(
    "LOG_FILE",
    "request_logs.jsonl"
)

# Illustrative local cost estimate.
# This is NOT an actual Ollama API charge.
COST_PER_1K_TOKENS = float(
    os.getenv("COST_PER_1K_TOKENS", "0.001")
)


# ============================================================
# APPLICATION
# ============================================================

app = FastAPI(
    title="Indian Tax RAG API",
    version="1.0.0"
)


# ============================================================
# CACHE
# ============================================================

CACHE = {}
CACHE_LOCK = threading.Lock()


# ============================================================
# REQUEST MODEL
# ============================================================

class ChatRequest(BaseModel):
    query: str = Field(
        ...,
        min_length=3,
        max_length=500,
        description="User tax-related question"
    )


# ============================================================
# TAX KNOWLEDGE BASE
# ============================================================

tax_documents = [
    {
        "id": "tax_001",
        "title": "Section 80C",
        "text": """
Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year.
"""
    },
    {
        "id": "tax_002",
        "title": "Form 16",
        "text": """
Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return.
"""
    },
    {
        "id": "tax_003",
        "title": "HRA",
        "text": """
House Rent Allowance, commonly called HRA, is an allowance that may be
provided by an employer as part of salary. Eligible salaried taxpayers
may claim an exemption related to HRA subject to applicable conditions.
"""
    },
    {
        "id": "tax_004",
        "title": "Tax Deduction",
        "text": """
A tax deduction reduces the amount of income considered for taxation.
Eligible deductions are generally claimed from gross total income while
calculating taxable income, subject to the applicable rules.
"""
    },
    {
        "id": "tax_005",
        "title": "Tax Exemption",
        "text": """
A tax exemption generally means that specified income or a specified
portion of income is not included in taxable income when the applicable
conditions are satisfied.
"""
    },
    {
        "id": "tax_006",
        "title": "TDS",
        "text": """
Tax Deducted at Source, or TDS, is a mechanism under which tax is
deducted at the time of specified payments and deposited with the
government on behalf of the recipient.
"""
    },
    {
        "id": "tax_007",
        "title": "Income Tax Return",
        "text": """
An Income Tax Return, or ITR, is a return filed by a taxpayer to report
income, deductions, taxes paid and other required information for the
relevant assessment year.
"""
    },
    {
        "id": "tax_008",
        "title": "Capital Gains",
        "text": """
Capital gains generally arise when a taxpayer transfers a capital asset
for consideration and realizes a gain or loss. The applicable tax
treatment depends on the type of asset, holding period and relevant
tax rules.
"""
    },
    {
        "id": "tax_009",
        "title": "Advance Tax",
        "text": """
Advance tax is income tax paid during the financial year in installments
when the taxpayer meets the conditions requiring payment of advance tax.
The applicable installments and rules depend on the taxpayer's situation.
"""
    },
    {
        "id": "tax_010",
        "title": "New Tax Regime",
        "text": """
India provides different income tax regimes. The applicable deductions,
exemptions and tax rates can differ between regimes and may change
between financial years. Taxpayers should verify the rules applicable
to the relevant year.
"""
    },
    {
        "id": "tax_011",
        "title": "Old Tax Regime",
        "text": """
The old tax regime allows various deductions and exemptions subject to
eligibility conditions. The actual tax treatment depends on the
financial year and taxpayer circumstances.
"""
    },
    {
        "id": "tax_012",
        "title": "GST",
        "text": """
Goods and Services Tax, or GST, is an indirect tax applied to the supply
of goods and services in India. GST is separate from personal income tax.
"""
    }
]


# ============================================================
# CHROMADB CONNECTION
# ============================================================

chroma_host = os.getenv("CHROMA_HOST")

if chroma_host:
    chroma_client = chromadb.HttpClient(
        host=chroma_host,
        port=8000
    )
else:
    chroma_client = chromadb.PersistentClient(
        path="./chroma_data"
    )

collection = chroma_client.get_or_create_collection(
    name="indian_tax_knowledge"
)


# ============================================================
# SEED KNOWLEDGE BASE
# ============================================================

collection.upsert(
    ids=[document["id"] for document in tax_documents],
    documents=[document["text"] for document in tax_documents],
    metadatas=[
        {"title": document["title"]}
        for document in tax_documents
    ]
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def estimate_tokens(text: str) -> int:
    """
    Simple token estimate used for observability.
    This is an approximation, not a tokenizer calculation.
    """

    if not text:
        return 0

    return max(1, len(text.split()))


def estimate_cost(
    input_tokens: int,
    output_tokens: int
) -> float:

    total_tokens = input_tokens + output_tokens

    return round(
        (total_tokens / 1000) * COST_PER_1K_TOKENS,
        6
    )


def retrieve_context(
    query: str,
    number_of_results: int = 3
):

    results = collection.query(
        query_texts=[query],
        n_results=number_of_results
    )

    return results.get(
        "documents",
        [[]]
    )[0]


def call_llm(
    query: str,
    context: list[str]
) -> str:

    context_text = "\n\n".join(context)

    prompt = f"""
You are an Indian tax information assistant.

Answer the user's question using ONLY the
provided knowledge context.

If the answer is not available in the context,
say that the information is not available.

Do not invent tax rates, thresholds or deadlines.

Question:
{query}

Knowledge context:
{context_text}

Answer:
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.1
        }
    }

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    result = response.json()

    answer = result.get(
        "response",
        ""
    ).strip()

    if not answer:
        raise RuntimeError(
            "LLM returned an empty response."
        )

    return answer


def write_log(
    query: str,
    latency_ms: float,
    input_tokens: int,
    output_tokens: int,
    cache_hit: bool
):

    estimated_cost = estimate_cost(
        input_tokens,
        output_tokens
    )

    log_entry = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),

        "query": query,

        "latency_ms": round(
            latency_ms,
            2
        ),

        "input_tokens_estimated": input_tokens,

        "output_tokens_estimated": output_tokens,

        "total_tokens_estimated":
            input_tokens + output_tokens,

        "estimated_cost_usd":
            estimated_cost,

        "model": MODEL_NAME,

        "cache_hit": cache_hit
    }

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as file:

        file.write(
            json.dumps(log_entry)
            + "\n"
        )


# ============================================================
# HEALTH CHECK
# ============================================================

@app.get("/health")
def health():

    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "cache_entries": len(CACHE),
        "documents": collection.count()
    }


# ============================================================
# CHAT ENDPOINT
# ============================================================

@app.post("/chat")
def chat(request: ChatRequest):

    query = request.query.strip()

    if not query:

        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    start_time = time.perf_counter()

    normalized_query = query.lower()

    # --------------------------------------------------------
    # CHECK CACHE
    # --------------------------------------------------------

    with CACHE_LOCK:

        if normalized_query in CACHE:

            answer = CACHE[
                normalized_query
            ]

            cache_hit = True

        else:

            cache_hit = False

            # ------------------------------------------------
            # RETRIEVE CONTEXT
            # ------------------------------------------------

            context = retrieve_context(
                query
            )

            if not context:

                raise HTTPException(
                    status_code=404,
                    detail="No relevant information found."
                )

            # ------------------------------------------------
            # GENERATE ANSWER
            # ------------------------------------------------

            answer = call_llm(
                query,
                context
            )

            CACHE[
                normalized_query
            ] = answer

    # --------------------------------------------------------
    # OBSERVABILITY
    # --------------------------------------------------------

    latency_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    input_tokens = estimate_tokens(
        query
    )

    output_tokens = estimate_tokens(
        answer
    )

    write_log(
        query=query,
        latency_ms=latency_ms,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_hit=cache_hit
    )

    # --------------------------------------------------------
    # SSE STREAMING
    # --------------------------------------------------------

    def generate_stream() -> Generator[str, None, None]:

        words = answer.split()

        for word in words:

            yield (
                "data: "
                + word
                + "\n\n"
            )

            time.sleep(0.015)

        yield "data: [DONE]\n\n"

    return StreamingResponse(
        generate_stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive"
        }
    )


# ============================================================
# STARTUP MESSAGE
# ============================================================

print(
    f"Indian Tax RAG API initialized "
    f"with {collection.count()} documents."
)

Writing app.py


In [ ]:
# ============================================================
# CELL 13 - START FASTAPI SERVER
# ============================================================

import subprocess
import time
import requests

print("Starting FastAPI server...")

api_process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=10
    )

    print("API status code:", response.status_code)
    print("API response:")
    print(response.json())

except Exception as e:
    print("Could not connect to FastAPI.")
    print("Error:", e)

Starting FastAPI server...
API status code: 200
API response:
{'status': 'healthy', 'model': 'qwen2.5:1.5b', 'cache_entries': 0, 'documents': 12}


In [ ]:
# ============================================================
# CELL 14 - TEST /CHAT ENDPOINT
# ============================================================

import requests

query = "What is Form 16?"

print("Sending query:")
print(query)

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    timeout=120
)

print("\nHTTP Status:", response.status_code)
print("\nStreaming response:")
print("=" * 70)

for line in response.text.splitlines():
    if line.startswith("data:"):
        text = line.replace("data:", "", 1).strip()

        if text != "[DONE]":
            print(text, end=" ")

print("\n" + "=" * 70)
print("Chat request completed successfully.")

Sending query:
What is Form 16?

HTTP Status: 200

Streaming response:
Form 16 is a certificate issued by an employer to an employee showing salary income and tax deducted at source from salary during the relevant financial year. It is useful when preparing an income tax return. 
Chat request completed successfully.


In [ ]:
# ============================================================
# CELL 15 - TEST CACHE PERFORMANCE
# ============================================================

import requests
import time

query = "What is Form 16?"


def send_request(question):
    start_time = time.perf_counter()

    response = requests.post(
        "http://127.0.0.1:8000/chat",
        json={"query": question},
        timeout=120
    )

    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000

    return response, latency_ms


# ------------------------------------------------------------
# FIRST REQUEST - EXPECTED CACHE MISS
# ------------------------------------------------------------

print("First request...")
response_1, latency_1 = send_request(query)

print("Status:", response_1.status_code)
print("Latency: {:.2f} ms".format(latency_1))


# ------------------------------------------------------------
# SECOND REQUEST - EXPECTED CACHE HIT
# ------------------------------------------------------------

print("\nSecond request...")
response_2, latency_2 = send_request(query)

print("Status:", response_2.status_code)
print("Latency: {:.2f} ms".format(latency_2))


# ------------------------------------------------------------
# COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE PERFORMANCE")
print("=" * 70)

print("First request : {:.2f} ms".format(latency_1))
print("Second request: {:.2f} ms".format(latency_2))

if latency_2 < latency_1:
    improvement = (
        (latency_1 - latency_2)
        / latency_1
    ) * 100

    print(
        "Latency improvement: {:.2f}%".format(
            improvement
        )
    )

print("\nCache test completed.")

First request...
Status: 200
Latency: 556.87 ms

Second request...
Status: 200
Latency: 557.99 ms

CACHE PERFORMANCE
First request : 556.87 ms
Second request: 557.99 ms

Cache test completed.


In [ ]:
# ============================================================
# CELL 16 - CHECK OBSERVABILITY LOGS
# ============================================================

import json
import os

log_file = "request_logs.jsonl"

if not os.path.exists(log_file):
    print("Log file was not created yet.")
else:
    print("Observability log found.")
    print("=" * 80)

    with open(log_file, "r", encoding="utf-8") as file:
        log_lines = file.readlines()

    print("Total logged requests:", len(log_lines))

    print("\nLatest log entries:")
    print("=" * 80)

    for line in log_lines[-5:]:
        entry = json.loads(line)

        print(
            f"Timestamp       : {entry['timestamp']}\n"
            f"Query           : {entry['query']}\n"
            f"Latency         : {entry['latency_ms']} ms\n"
            f"Input tokens    : {entry['input_tokens_estimated']}\n"
            f"Output tokens   : {entry['output_tokens_estimated']}\n"
            f"Total tokens    : {entry['total_tokens_estimated']}\n"
            f"Estimated cost  : ${entry['estimated_cost_usd']}\n"
            f"Model           : {entry['model']}\n"
            f"Cache hit       : {entry['cache_hit']}\n"
        )
        print("-" * 80)

Observability log found.
Total logged requests: 3

Latest log entries:
Timestamp       : 2026-09-08T13:33:18.733733+00:00
Query           : What is Form 16?
Latency         : 749.91 ms
Input tokens    : 4
Output tokens   : 36
Total tokens    : 40
Estimated cost  : $4e-05
Model           : qwen2.5:1.5b
Cache hit       : False

--------------------------------------------------------------------------------
Timestamp       : 2026-09-08T13:33:19.303062+00:00
Query           : What is Form 16?
Latency         : 0.0 ms
Input tokens    : 4
Output tokens   : 36
Total tokens    : 40
Estimated cost  : $4e-05
Model           : qwen2.5:1.5b
Cache hit       : True

--------------------------------------------------------------------------------
Timestamp       : 2026-09-08T13:33:19.859554+00:00
Query           : What is Form 16?
Latency         : 0.0 ms
Input tokens    : 4
Output tokens   : 36
Total tokens    : 40
Estimated cost  : $4e-05
Model           : qwen2.5:1.5b
Cache hit       : True

----

In [ ]:
# ============================================================
# CELL 17 - CREATE DEEPEVAL TEST CASES
# ============================================================

evaluation_cases = [
    {
        "input": "What is Section 80C?",
        "expected": (
            "Section 80C provides eligible taxpayers with deductions "
            "for specified investments and payments subject to applicable rules."
        )
    },
    {
        "input": "What is Form 16?",
        "expected": (
            "Form 16 is a certificate issued by an employer showing "
            "salary income and tax deducted at source."
        )
    },
    {
        "input": "What is HRA?",
        "expected": (
            "HRA is House Rent Allowance and eligible taxpayers may "
            "receive an exemption subject to applicable conditions."
        )
    },
    {
        "input": "What is TDS?",
        "expected": (
            "TDS means Tax Deducted at Source and is a mechanism where "
            "tax is deducted from specified payments."
        )
    },
    {
        "input": "What is an income tax return?",
        "expected": (
            "An income tax return reports income, deductions, taxes paid "
            "and other required information."
        )
    },
    {
        "input": "What are capital gains?",
        "expected": (
            "Capital gains generally arise when a taxpayer transfers "
            "a capital asset and realizes a gain or loss."
        )
    },
    {
        "input": "What is advance tax?",
        "expected": (
            "Advance tax is income tax paid during the financial year "
            "in installments when applicable conditions are satisfied."
        )
    },
    {
        "input": "What is the new tax regime?",
        "expected": (
            "The new tax regime is one of the income tax regimes and "
            "its deductions, exemptions and rates may differ from other regimes."
        )
    },
    {
        "input": "What is the old tax regime?",
        "expected": (
            "The old tax regime allows various deductions and exemptions "
            "subject to eligibility conditions."
        )
    },
    {
        "input": "What is GST?",
        "expected": (
            "GST is an indirect tax applied to the supply of goods "
            "and services in India."
        )
    }
]

print("DeepEval test cases created:", len(evaluation_cases))

print("\nTest cases:")
print("=" * 80)

for number, case in enumerate(evaluation_cases, start=1):
    print(f"{number}. {case['input']}")

DeepEval test cases created: 10

Test cases:
1. What is Section 80C?
2. What is Form 16?
3. What is HRA?
4. What is TDS?
5. What is an income tax return?
6. What are capital gains?
7. What is advance tax?
8. What is the new tax regime?
9. What is the old tax regime?
10. What is GST?


In [ ]:
# ============================================================
# CELL 18 - GENERATE ANSWERS FOR DEEPEVAL
# ============================================================

import requests

test_results = []

print("Running the 10 evaluation questions...")
print("=" * 80)

for number, case in enumerate(evaluation_cases, start=1):

    question = case["input"]

    print(f"\nTest {number}/10: {question}")

    try:
        response = requests.post(
            "http://127.0.0.1:8000/chat",
            json={"query": question},
            timeout=120
        )

        if response.status_code != 200:
            print("Request failed:", response.text)
            continue

        # ----------------------------------------------------
        # Extract the SSE response
        # ----------------------------------------------------

        answer_parts = []

        for line in response.text.splitlines():

            if line.startswith("data:"):

                text = line.replace(
                    "data:",
                    "",
                    1
                ).strip()

                if text and text != "[DONE]":
                    answer_parts.append(text)

        answer = " ".join(answer_parts)

        # ----------------------------------------------------
        # Retrieve the same context from ChromaDB
        # ----------------------------------------------------

        retrieval_results = collection.query(
            query_texts=[question],
            n_results=3
        )

        retrieval_context = retrieval_results[
            "documents"
        ][0]

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        test_results.append(
            {
                "input": question,
                "actual_output": answer,
                "expected_output": case["expected"],
                "retrieval_context": retrieval_context
            }
        )

        print("Status: SUCCESS")
        print("Answer:", answer[:200], "...")

    except Exception as error:

        print("Error:", error)


print("\n" + "=" * 80)
print("Evaluation data collection completed.")
print("Successful test cases:", len(test_results))

Running the 10 evaluation questions...

Test 1/10: What is Section 80C?
Status: SUCCESS
Answer: Section 80C allows eligible individual taxpayers and Hindu Undivided Families to claim deductions for certain specified investments and payments, subject to the conditions and limits applicable for th ...

Test 2/10: What is Form 16?
Status: SUCCESS
Answer: Form 16 is a certificate issued by an employer to an employee showing salary income and tax deducted at source from salary during the relevant financial year. It is useful when preparing an income tax ...

Test 3/10: What is HRA?
Status: SUCCESS
Answer: HRA, or House Rent Allowance, is an allowance provided by an employer as part of salary. Eligible salaried taxpayers and Hindu Undivided Families can claim an exemption related to HRA subject to appli ...

Test 4/10: What is TDS?
Status: SUCCESS
Answer: TDS stands for Tax Deducted at Source. ...

Test 5/10: What is an income tax return?
Status: SUCCESS
Answer: An income tax return is a doc

In [ ]:
# ============================================================
# CELL 19 - CONFIGURE DEEPEVAL METRICS
# ============================================================

from deepeval.models import OllamaModel

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric
)


# Use the same local Qwen model as the evaluation judge
evaluation_model = OllamaModel(
    model="qwen2.5:1.5b",
    base_url="http://127.0.0.1:11434",
    temperature=0
)


# ------------------------------------------------------------
# CREATE THE THREE REQUIRED METRICS
# ------------------------------------------------------------

answer_relevancy = AnswerRelevancyMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

faithfulness = FaithfulnessMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)


metrics = [
    answer_relevancy,
    faithfulness,
    contextual_precision
]


print("DeepEval configured successfully.")
print("\nMetrics:")
print("1. Answer Relevancy")
print("2. Faithfulness")
print("3. Contextual Precision")

print("\nEvaluation model:")
print("qwen2.5:1.5b via Ollama")

DeepEval configured successfully.

Metrics:
1. Answer Relevancy
2. Faithfulness
3. Contextual Precision

Evaluation model:
qwen2.5:1.5b via Ollama


In [ ]:
# ============================================================
# CELL 19A - INSTALL OLLAMA PYTHON PACKAGE
# ============================================================

!pip -q install ollama

print("Python Ollama package installed successfully.")

Python Ollama package installed successfully.


In [ ]:
# ============================================================
# CELL 19 - CONFIGURE DEEPEVAL METRICS
# ============================================================

from deepeval.models import OllamaModel

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric
)


# Use the same local Qwen model as the evaluation judge
evaluation_model = OllamaModel(
    model="qwen2.5:1.5b",
    base_url="http://127.0.0.1:11434",
    temperature=0
)


# ------------------------------------------------------------
# CREATE THE THREE REQUIRED METRICS
# ------------------------------------------------------------

answer_relevancy = AnswerRelevancyMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

faithfulness = FaithfulnessMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)


metrics = [
    answer_relevancy,
    faithfulness,
    contextual_precision
]


print("DeepEval configured successfully.")
print("\nMetrics:")
print("1. Answer Relevancy")
print("2. Faithfulness")
print("3. Contextual Precision")

print("\nEvaluation model:")
print("qwen2.5:1.5b via Ollama")

DeepEval configured successfully.

Metrics:
1. Answer Relevancy
2. Faithfulness
3. Contextual Precision

Evaluation model:
qwen2.5:1.5b via Ollama


In [ ]:
# ============================================================
# CELL 19A - INSTALL OLLAMA PYTHON PACKAGE
# ============================================================

!pip -q install ollama

print("Python Ollama package installed successfully.")

Python Ollama package installed successfully.


In [ ]:
# ============================================================
# CELL 20 - CREATE DEEPEVAL TEST CASE OBJECTS
# ============================================================

from deepeval.test_case import LLMTestCase

deepeval_cases = []

for result in test_results:

    test_case = LLMTestCase(
        input=result["input"],
        actual_output=result["actual_output"],
        expected_output=result["expected_output"],
        retrieval_context=result["retrieval_context"]
    )

    deepeval_cases.append(test_case)


print("DeepEval test cases created successfully.")
print("Total test cases:", len(deepeval_cases))

print("\nFirst test case:")
print("-" * 70)
print("Input:", deepeval_cases[0].input)
print("Actual output:", deepeval_cases[0].actual_output)
print("Expected output:", deepeval_cases[0].expected_output)
print(
    "Retrieved contexts:",
    len(deepeval_cases[0].retrieval_context)
)

DeepEval test cases created successfully.
Total test cases: 10

First test case:
----------------------------------------------------------------------
Input: What is Section 80C?
Actual output: Section 80C allows eligible individual taxpayers and Hindu Undivided Families to claim deductions for certain specified investments and payments, subject to the conditions and limits applicable for the relevant financial year.
Expected output: Section 80C provides eligible taxpayers with deductions for specified investments and payments subject to applicable rules.
Retrieved contexts: 3


In [ ]:
# ============================================================
# CELL 20A - INSTALL DEEPEVAL
# ============================================================

!pip -q install deepeval

print("DeepEval installed successfully.")

DeepEval installed successfully.


In [ ]:
from deepeval.models import OllamaModel

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric
)

evaluation_model = OllamaModel(
    model="qwen2.5:1.5b",
    base_url="http://127.0.0.1:11434",
    temperature=0
)

answer_relevancy = AnswerRelevancyMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

faithfulness = FaithfulnessMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

metrics = [
    answer_relevancy,
    faithfulness,
    contextual_precision
]

print("DeepEval configured successfully.")
print("Metrics:")
print("1. Answer Relevancy")
print("2. Faithfulness")
print("3. Contextual Precision")
print("\nEvaluation model: qwen2.5:1.5b via Ollama")

DeepEval configured successfully.
Metrics:
1. Answer Relevancy
2. Faithfulness
3. Contextual Precision

Evaluation model: qwen2.5:1.5b via Ollama


In [ ]:
from deepeval.test_case import LLMTestCase

deepeval_cases = []

for result in test_results:

    test_case = LLMTestCase(
        input=result["input"],
        actual_output=result["actual_output"],
        expected_output=result["expected_output"],
        retrieval_context=result["retrieval_context"]
    )

    deepeval_cases.append(test_case)

print("DeepEval test cases created successfully.")
print("Total test cases:", len(deepeval_cases))

print("\nFirst test case:")
print("-" * 70)
print("Input:", deepeval_cases[0].input)
print("Actual output:", deepeval_cases[0].actual_output)
print("Expected output:", deepeval_cases[0].expected_output)
print("Retrieved contexts:", len(deepeval_cases[0].retrieval_context))

DeepEval test cases created successfully.
Total test cases: 10

First test case:
----------------------------------------------------------------------
Input: What is Section 80C?
Actual output: Section 80C allows eligible individual taxpayers and Hindu Undivided Families to claim deductions for certain specified investments and payments, subject to the conditions and limits applicable for the relevant financial year.
Expected output: Section 80C provides eligible taxpayers with deductions for specified investments and payments subject to applicable rules.
Retrieved contexts: 3


In [ ]:
# ============================================================
# CELL 21 - RUN DEEPEVAL EVALUATION
# ============================================================

from deepeval import evaluate

print("=" * 80)
print("STARTING DEEPEVAL EVALUATION")
print("=" * 80)

print("\nTest cases:", len(deepeval_cases))
print("Metrics:", len(metrics))
print("\nThis may take a few minutes...")
print("Qwen2.5 will be used as the local evaluation model.\n")


evaluation_results = evaluate(
    test_cases=deepeval_cases,
    metrics=metrics
)

print("\n" + "=" * 80)
print("DEEPEVAL EVALUATION COMPLETED")
print("=" * 80)

STARTING DEEPEVAL EVALUATION

Test cases: 10
Metrics: 3

This may take a few minutes...
Qwen2.5 will be used as the local evaluation model.



✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen2.5:1.5b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen2.5:1.5b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen2.5:1.5b (Ollama), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is Section 80C?                                                                 │
│  │     Actual Output:      Section 80C allows eligible individual taxpayers and Hindu Undivided Families to     │
│  │                         claim deductions for certain specified investments and payments, subject to the      │
│  │                         conditions and limits applicable for the relevant financial year.                    │
│  │     Expected Output:    Section 80C provides eligible taxpayers with deductions for specified investments    │
│  │                         and payments subject to applicable rules.                                            │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.50      │ The score is 1.00 because the input asks for in...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.50      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.50      │ The score is 0.00 because the relevant nodes are      │
│              │                      │       │           │ ranked lower than the irrelevant nodes. The input     │
│              │                      │       │           │ 'Section 80C' is not mentioned in any of the          │
│              │                      │       │           │ retrieval contexts, which are all irrelevant.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What is Form 16?                                                                     │
│  │     Actual Output:      Form 16 is a certificate issued by an employer to an employee showing salary         │
│  │                         income and tax deducted at source from salary during the relevant financial year.    │
│  │                         It is useful when preparing an income tax return.                                    │
│  │     Expected Output:    Form 16 is a certificate issued by an employer showing salary income and tax         │
│  │                         deducted at source.                                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇

⚠ WARNING: No hyperparameters logged.
» ]8;id=783226;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 52.34s | token cost: None)
» Test Results (10 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 10

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


DEEPEVAL EVALUATION COMPLETED


In [ ]:
# ============================================================
# CELL 22 - INSPECT RAG RETRIEVAL QUALITY
# ============================================================

print("=" * 80)
print("CHECKING CHROMADB RETRIEVAL FOR ALL 10 QUESTIONS")
print("=" * 80)

for number, case in enumerate(evaluation_cases, start=1):

    question = case["input"]

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    documents = results["documents"][0]
    distances = results.get("distances", [[]])[0]

    print(f"\nTEST {number}: {question}")
    print("-" * 80)

    for rank, document in enumerate(documents, start=1):

        distance = (
            distances[rank - 1]
            if rank - 1 < len(distances)
            else "N/A"
        )

        print(f"\nRank {rank}")
        print(f"Distance: {distance}")
        print("Context:")
        print(document.strip()[:300])

print("\n" + "=" * 80)
print("RETRIEVAL INSPECTION COMPLETED")
print("=" * 80)

CHECKING CHROMADB RETRIEVAL FOR ALL 10 QUESTIONS

TEST 1: What is Section 80C?
--------------------------------------------------------------------------------

Rank 1
Distance: 0.8120957016944885
Context:
Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year.

Rank 2
Distance: 1.5137168169021606
Context:
House Rent Allowance, commonly called HRA, is an allowance that may be
provided by an employer as part of salary. Eligible salaried taxpayers
may claim an exemption related to HRA subject to applicable conditions.

Rank 3
Distance: 1.563759207725525
Context:
Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return.

TEST 2: What is Form 16?
---------------------------

In [ ]:
# ============================================================
# CELL 23 - IMPROVE RAG KNOWLEDGE DOCUMENTS
# ============================================================

# Update the tax documents with clearer keywords and descriptions.
# This improves semantic retrieval for questions such as
# "What is the new tax regime?"

improved_tax_documents = [
    {
        "id": "tax_01",
        "text": """Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year."""
    },
    {
        "id": "tax_02",
        "text": """Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return."""
    },
    {
        "id": "tax_03",
        "text": """House Rent Allowance, commonly called HRA, is an allowance that may be
provided by an employer as part of salary. Eligible salaried taxpayers
may claim an exemption related to HRA subject to applicable conditions."""
    },
    {
        "id": "tax_04",
        "text": """Tax Deducted at Source, or TDS, is a mechanism under which tax is
deducted at the time of specified payments and deposited with the
government on behalf of the recipient."""
    },
    {
        "id": "tax_05",
        "text": """An Income Tax Return, or ITR, is a return filed by a taxpayer to report
income, deductions, taxes paid and other required information for the
relevant assessment year."""
    },
    {
        "id": "tax_06",
        "text": """Capital gains generally arise when a taxpayer transfers a capital asset
for consideration and realizes a gain or loss. The applicable tax
treatment depends on the type of asset, holding period and relevant
tax rules."""
    },
    {
        "id": "tax_07",
        "text": """Advance tax is income tax paid during the financial year in installments
when the taxpayer meets the conditions requiring payment of advance tax.
The applicable installments and rules depend on the taxpayer's situation."""
    },
    {
        "id": "tax_08",
        "text": """The new tax regime is one of India's income tax regimes. Under the
new tax regime, applicable tax rates, deductions and exemptions differ
from those available under the old tax regime. The exact rules and tax
rates may change between financial years, so taxpayers should verify
the rules applicable to the relevant assessment year."""
    },
    {
        "id": "tax_09",
        "text": """The old tax regime is an income tax regime that allows various
deductions and exemptions subject to eligibility conditions. The actual
tax treatment depends on the financial year and taxpayer circumstances."""
    },
    {
        "id": "tax_10",
        "text": """Goods and Services Tax, or GST, is an indirect tax applied to the supply
of goods and services in India. GST is separate from personal income tax."""
    }
]

# Replace the existing ChromaDB documents.
collection.upsert(
    ids=[doc["id"] for doc in improved_tax_documents],
    documents=[doc["text"] for doc in improved_tax_documents]
)

print("=" * 80)
print("CHROMADB KNOWLEDGE BASE UPDATED")
print("=" * 80)
print(f"Documents indexed: {collection.count()}")
print("New tax regime document updated with explicit keywords.")
print("=" * 80)

CHROMADB KNOWLEDGE BASE UPDATED
Documents indexed: 22
New tax regime document updated with explicit keywords.


In [ ]:
# ============================================================
# CELL 24 - INSPECT CHROMADB DOCUMENT IDS
# ============================================================

print("=" * 80)
print("CURRENT CHROMADB DOCUMENTS")
print("=" * 80)

all_documents = collection.get()

print(f"Total documents: {len(all_documents['ids'])}")
print("\nDocument IDs:")

for document_id in all_documents["ids"]:
    print("-", document_id)

print("=" * 80)

CURRENT CHROMADB DOCUMENTS
Total documents: 22

Document IDs:
- tax_001
- tax_002
- tax_003
- tax_004
- tax_005
- tax_006
- tax_007
- tax_008
- tax_009
- tax_010
- tax_011
- tax_012
- tax_01
- tax_02
- tax_03
- tax_04
- tax_05
- tax_06
- tax_07
- tax_08
- tax_09
- tax_10


In [ ]:
# ============================================================
# CELL 25 - CLEAN AND REBUILD CHROMADB COLLECTION
# ============================================================

print("=" * 80)
print("RESETTING CHROMADB KNOWLEDGE BASE")
print("=" * 80)

# Delete the existing collection completely.
collection_name = "indian_tax_knowledge"

try:
    chroma_client.delete_collection(name=collection_name)
    print("Old collection deleted successfully.")
except Exception as e:
    print("Collection deletion message:", e)

# Create a fresh collection.
collection = chroma_client.get_or_create_collection(
    name=collection_name
)

# Insert exactly the 10 improved documents.
collection.upsert(
    ids=[doc["id"] for doc in improved_tax_documents],
    documents=[doc["text"] for doc in improved_tax_documents]
)

print(f"\nNew collection created: {collection_name}")
print(f"Documents indexed: {collection.count()}")

print("\nDocument IDs:")
for document_id in collection.get()["ids"]:
    print("-", document_id)

print("=" * 80)
print("CHROMADB RESET COMPLETED")
print("=" * 80)

RESETTING CHROMADB KNOWLEDGE BASE
Old collection deleted successfully.

New collection created: indian_tax_knowledge
Documents indexed: 10

Document IDs:
- tax_01
- tax_02
- tax_03
- tax_04
- tax_05
- tax_06
- tax_07
- tax_08
- tax_09
- tax_10
CHROMADB RESET COMPLETED


In [ ]:
# ============================================================
# CELL 26 - VERIFY NEW TAX REGIME RETRIEVAL
# ============================================================

question = "What is the new tax regime?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

print("=" * 80)
print("NEW TAX REGIME RETRIEVAL TEST")
print("=" * 80)

for rank, document in enumerate(results["documents"][0], start=1):
    distance = results["distances"][0][rank - 1]

    print(f"\nRank {rank}")
    print(f"Distance: {distance}")
    print("Context:")
    print(document.strip())

print("\n" + "=" * 80)

NEW TAX REGIME RETRIEVAL TEST

Rank 1
Distance: 0.40907353162765503
Context:
The old tax regime is an income tax regime that allows various
deductions and exemptions subject to eligibility conditions. The actual
tax treatment depends on the financial year and taxpayer circumstances.

Rank 2
Distance: 0.6717313528060913
Context:
The new tax regime is one of India's income tax regimes. Under the
new tax regime, applicable tax rates, deductions and exemptions differ
from those available under the old tax regime. The exact rules and tax
rates may change between financial years, so taxpayers should verify
the rules applicable to the relevant assessment year.

Rank 3
Distance: 1.0804411172866821
Context:
Advance tax is income tax paid during the financial year in installments
when the taxpayer meets the conditions requiring payment of advance tax.
The applicable installments and rules depend on the taxpayer's situation.



In [ ]:
# ============================================================
# CELL 27 - TEST KEYWORD-AWARE RETRIEVAL
# ============================================================

def keyword_aware_retrieval(question, n_results=3):
    """
    Combines ChromaDB semantic retrieval with a simple
    exact phrase boost for important tax-related terms.
    """

    results = collection.query(
        query_texts=[question],
        n_results=10
    )

    documents = results["documents"][0]
    distances = results["distances"][0]

    question_lower = question.lower()

    # Important phrases in our knowledge base
    important_terms = [
        "section 80c",
        "form 16",
        "hra",
        "tds",
        "income tax return",
        "capital gains",
        "advance tax",
        "new tax regime",
        "old tax regime",
        "gst"
    ]

    scored_results = []

    for document, distance in zip(documents, distances):

        document_lower = document.lower()

        # Convert distance into a similarity-like score.
        semantic_score = 1 / (1 + distance)

        # Exact phrase matching boost.
        keyword_boost = 0

        for term in important_terms:
            if term in question_lower and term in document_lower:
                keyword_boost += 1.0

        final_score = semantic_score + keyword_boost

        scored_results.append(
            {
                "document": document,
                "distance": distance,
                "semantic_score": semantic_score,
                "keyword_boost": keyword_boost,
                "final_score": final_score
            }
        )

    # Highest final score first.
    scored_results.sort(
        key=lambda item: item["final_score"],
        reverse=True
    )

    return scored_results[:n_results]


question = "What is the new tax regime?"

retrieved = keyword_aware_retrieval(question)

print("=" * 80)
print("KEYWORD-AWARE RETRIEVAL TEST")
print("=" * 80)

for rank, item in enumerate(retrieved, start=1):

    print(f"\nRank {rank}")
    print(f"Semantic score: {item['semantic_score']:.3f}")
    print(f"Keyword boost: {item['keyword_boost']:.3f}")
    print(f"Final score: {item['final_score']:.3f}")
    print("Context:")
    print(item["document"].strip())

print("\n" + "=" * 80)

KEYWORD-AWARE RETRIEVAL TEST

Rank 1
Semantic score: 0.598
Keyword boost: 1.000
Final score: 1.598
Context:
The new tax regime is one of India's income tax regimes. Under the
new tax regime, applicable tax rates, deductions and exemptions differ
from those available under the old tax regime. The exact rules and tax
rates may change between financial years, so taxpayers should verify
the rules applicable to the relevant assessment year.

Rank 2
Semantic score: 0.710
Keyword boost: 0.000
Final score: 0.710
Context:
The old tax regime is an income tax regime that allows various
deductions and exemptions subject to eligibility conditions. The actual
tax treatment depends on the financial year and taxpayer circumstances.

Rank 3
Semantic score: 0.481
Keyword boost: 0.000
Final score: 0.481
Context:
Advance tax is income tax paid during the financial year in installments
when the taxpayer meets the conditions requiring payment of advance tax.
The applicable installments and rules depend on t

In [ ]:
# ============================================================
# CELL 28 - VERIFY KEYWORD-AWARE RETRIEVAL FOR ALL TEST CASES
# ============================================================

print("=" * 80)
print("KEYWORD-AWARE RETRIEVAL - ALL 10 TEST CASES")
print("=" * 80)

retrieval_results = []

for number, case in enumerate(evaluation_cases, start=1):

    question = case["input"]

    retrieved = keyword_aware_retrieval(
        question,
        n_results=3
    )

    top_context = retrieved[0]["document"]

    print(f"\nTEST {number}: {question}")
    print("-" * 80)
    print("Top retrieved context:")
    print(top_context.strip())

    retrieval_results.append(
        {
            "test_number": number,
            "question": question,
            "top_context": top_context
        }
    )

print("\n" + "=" * 80)
print("ALL 10 RETRIEVAL TESTS COMPLETED")
print("=" * 80)

KEYWORD-AWARE RETRIEVAL - ALL 10 TEST CASES

TEST 1: What is Section 80C?
--------------------------------------------------------------------------------
Top retrieved context:
Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year.

TEST 2: What is Form 16?
--------------------------------------------------------------------------------
Top retrieved context:
Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return.

TEST 3: What is HRA?
--------------------------------------------------------------------------------
Top retrieved context:
House Rent Allowance, commonly called HRA, is an allowance that may be
provided by an employer as part of salary. Eligible salarie

In [ ]:
# ============================================================
# CELL 29 - GENERATE ANSWERS USING IMPROVED RETRIEVAL
# ============================================================

print("=" * 80)
print("GENERATING RAG ANSWERS WITH KEYWORD-AWARE RETRIEVAL")
print("=" * 80)

improved_test_results = []

for number, case in enumerate(evaluation_cases, start=1):

    question = case["input"]

    # Retrieve the best contexts using our improved retrieval.
    retrieved = keyword_aware_retrieval(
        question,
        n_results=3
    )

    contexts = [
        item["document"]
        for item in retrieved
    ]

    context_text = "\n\n".join(contexts)

    # Use the same Ollama model used by the API.
    prompt = f"""
You are a helpful Indian tax information assistant.

Answer the user's question using ONLY the information provided
in the context below.

If the answer is not available in the context, say:
"I don't have enough information in the provided context."

Context:
{context_text}

Question:
{question}

Answer:
"""

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.1
            }
        },
        timeout=120
    )

    response.raise_for_status()

    answer = response.json()["response"].strip()

    improved_test_results.append(
        {
            "input": question,
            "actual_output": answer,
            "expected_output": case["expected_output"],
            "retrieval_context": contexts
        }
    )

    print(f"\nTEST {number}")
    print(f"Question: {question}")
    print(f"Answer: {answer}")

print("\n" + "=" * 80)
print(f"SUCCESSFUL TEST CASES: {len(improved_test_results)}")
print("=" * 80)

GENERATING RAG ANSWERS WITH KEYWORD-AWARE RETRIEVAL


NameError: name 'OLLAMA_BASE_URL' is not defined

In [ ]:
# ============================================================
# CELL 29A - RESTORE OLLAMA CONFIGURATION
# ============================================================

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MODEL_NAME = "qwen2.5:1.5b"

print("=" * 80)
print("OLLAMA CONFIGURATION RESTORED")
print("=" * 80)

print(f"Ollama URL : {OLLAMA_BASE_URL}")
print(f"Model      : {MODEL_NAME}")

# Verify that Ollama is reachable.
response = requests.get(
    f"{OLLAMA_BASE_URL}/api/tags",
    timeout=30
)

response.raise_for_status()

models = response.json().get("models", [])

print(f"Available models: {len(models)}")

for model in models:
    print("-", model.get("name"))

print("=" * 80)

OLLAMA CONFIGURATION RESTORED
Ollama URL : http://127.0.0.1:11434
Model      : qwen2.5:1.5b
Available models: 1
- qwen2.5:1.5b


In [ ]:
# ============================================================
# CELL 29 - GENERATE ANSWERS USING IMPROVED RETRIEVAL
# ============================================================

print("=" * 80)
print("GENERATING RAG ANSWERS WITH KEYWORD-AWARE RETRIEVAL")
print("=" * 80)

improved_test_results = []

for number, case in enumerate(evaluation_cases, start=1):

    question = case["input"]

    # Retrieve the best contexts using our improved retrieval.
    retrieved = keyword_aware_retrieval(
        question,
        n_results=3
    )

    contexts = [
        item["document"]
        for item in retrieved
    ]

    context_text = "\n\n".join(contexts)

    # Use the same Ollama model used by the API.
    prompt = f"""
You are a helpful Indian tax information assistant.

Answer the user's question using ONLY the information provided
in the context below.

If the answer is not available in the context, say:
"I don't have enough information in the provided context."

Context:
{context_text}

Question:
{question}

Answer:
"""

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.1
            }
        },
        timeout=120
    )

    response.raise_for_status()

    answer = response.json()["response"].strip()

    improved_test_results.append(
        {
            "input": question,
            "actual_output": answer,
            "expected_output": case["expected_output"],
            "retrieval_context": contexts
        }
    )

    print(f"\nTEST {number}")
    print(f"Question: {question}")
    print(f"Answer: {answer}")

print("\n" + "=" * 80)
print(f"SUCCESSFUL TEST CASES: {len(improved_test_results)}")
print("=" * 80)

GENERATING RAG ANSWERS WITH KEYWORD-AWARE RETRIEVAL


KeyError: 'expected_output'

In [ ]:
# ============================================================
# CELL 29B - INSPECT EVALUATION CASE STRUCTURE
# ============================================================

print("=" * 80)
print("CHECKING EVALUATION CASE STRUCTURE")
print("=" * 80)

print(f"Number of evaluation cases: {len(evaluation_cases)}")

print("\nFirst evaluation case:")
print(evaluation_cases[0])

print("\nAvailable keys:")
print(list(evaluation_cases[0].keys()))

print("=" * 80)

CHECKING EVALUATION CASE STRUCTURE
Number of evaluation cases: 10

First evaluation case:
{'input': 'What is Section 80C?', 'expected': 'Section 80C provides eligible taxpayers with deductions for specified investments and payments subject to applicable rules.'}

Available keys:
['input', 'expected']


In [ ]:
# ============================================================
# CELL 29C - GENERATE ANSWERS WITH CORRECT EVALUATION KEYS
# ============================================================

print("=" * 80)
print("GENERATING RAG ANSWERS WITH KEYWORD-AWARE RETRIEVAL")
print("=" * 80)

improved_test_results = []

for number, case in enumerate(evaluation_cases, start=1):

    question = case["input"]

    # Retrieve the best contexts.
    retrieved = keyword_aware_retrieval(
        question,
        n_results=3
    )

    contexts = [
        item["document"]
        for item in retrieved
    ]

    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a helpful Indian tax information assistant.

Answer the user's question using ONLY the information provided
in the context below.

If the answer is not available in the context, say:
"I don't have enough information in the provided context."

Context:
{context_text}

Question:
{question}

Answer:
"""

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.1
            }
        },
        timeout=120
    )

    response.raise_for_status()

    answer = response.json()["response"].strip()

    improved_test_results.append(
        {
            "input": question,
            "actual_output": answer,
            "expected_output": case["expected"],
            "retrieval_context": contexts
        }
    )

    print(f"\nTEST {number}")
    print(f"Question: {question}")
    print(f"Answer: {answer}")

print("\n" + "=" * 80)
print(f"SUCCESSFUL TEST CASES: {len(improved_test_results)}")
print("=" * 80)

GENERATING RAG ANSWERS WITH KEYWORD-AWARE RETRIEVAL

TEST 1
Question: What is Section 80C?
Answer: Section 80C refers to the provision in the Indian Income Tax Act that allows eligible individuals and Hindu Undivided Families to claim deductions for certain specified investments and payments, subject to the conditions and limits applicable for the relevant financial year.

TEST 2
Question: What is Form 16?
Answer: Form 16 is a certificate issued by an employer to an employee showing salary income and tax deducted at source from salary during the relevant financial year.

TEST 3
Question: What is HRA?
Answer: HRA, or House Rent Allowance, is an allowance provided by an employer as part of salary to eligible salaried taxpayers.

TEST 4
Question: What is TDS?
Answer: TDS, or Tax Deducted at Source, is a mechanism under which tax is deducted at the time of specified payments and deposited with the government on behalf of the recipient.

TEST 5
Question: What is an income tax return?
Answer

In [ ]:
# ============================================================
# CELL 30 - CREATE DEEPEVAL TEST CASES
# ============================================================

from deepeval.test_case import LLMTestCase

deepeval_test_cases = []

for result in improved_test_results:

    test_case = LLMTestCase(
        input=result["input"],
        actual_output=result["actual_output"],
        expected_output=result["expected_output"],
        retrieval_context=result["retrieval_context"]
    )

    deepeval_test_cases.append(test_case)

print("=" * 80)
print("DEEPEVAL TEST CASES CREATED")
print("=" * 80)

print(f"Total test cases: {len(deepeval_test_cases)}")

print("\nFirst test case:")
print(f"Input: {deepeval_test_cases[0].input}")
print(f"Expected: {deepeval_test_cases[0].expected_output}")
print(f"Retrieved contexts: {len(deepeval_test_cases[0].retrieval_context)}")

print("=" * 80)

DEEPEVAL TEST CASES CREATED
Total test cases: 10

First test case:
Input: What is Section 80C?
Expected: Section 80C provides eligible taxpayers with deductions for specified investments and payments subject to applicable rules.
Retrieved contexts: 3


In [ ]:
# ============================================================
# CELL 31 - RUN DEEPEVAL EVALUATION
# ============================================================

from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric
)

print("=" * 80)
print("RUNNING DEEPEVAL EVALUATION")
print("=" * 80)

# Use the local Ollama model as the evaluation/judge model.
evaluation_model = OllamaModel(
    model="qwen2.5:1.5b",
    base_url="http://127.0.0.1:11434",
    temperature=0
)

# Metric 1: Answer Relevancy
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

# Metric 2: Faithfulness
faithfulness = FaithfulnessMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

# Metric 3: Contextual Precision
contextual_precision = ContextualPrecisionMetric(
    threshold=0.5,
    model=evaluation_model,
    include_reason=True
)

print("\nMetrics configured:")
print("1. Answer Relevancy")
print("2. Faithfulness")
print("3. Contextual Precision")

print("\nStarting evaluation...")
print("-" * 80)

evaluation_results = evaluate(
    deepeval_test_cases,
    metrics=[
        answer_relevancy,
        faithfulness,
        contextual_precision
    ]
)

print("\n" + "=" * 80)
print("DEEPEVAL EVALUATION COMPLETED")
print("=" * 80)

RUNNING DEEPEVAL EVALUATION

Metrics configured:
1. Answer Relevancy
2. Faithfulness
3. Contextual Precision

Starting evaluation...
--------------------------------------------------------------------------------


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen2.5:1.5b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen2.5:1.5b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen2.5:1.5b (Ollama), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is Section 80C?                                                                 │
│  │     Actual Output:      Section 80C refers to the provision in the Indian Income Tax Act that allows         │
│  │                         eligible individuals and Hindu Undivided Families to claim deductions for certain    │
│  │                         specified investments and payments, subject to the conditions and limits             │
│  │                         applicable for the relevant financial year.                                          │
│  │     Expected Output:    Section 80C provides eligible taxpayers with deductions for specified investments    │
│  │                         and payments subject to applicable rules.                                            │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.50      │ The score is 1.00 because the input asks for in...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.50      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.50      │ The score is 0.00 because all three retrieval         │
│              │                      │       │           │ contexts are irrelevant to the input 'What is         │
│              │                      │       │           │ Section 80C?'.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What is Form 16?                                                                     │
│  │     Actual Output:      Form 16 is a certificate issued by an employer to an employee showing salary         │
│  │                         income and tax deducted at source from salary during the relevant financial year.    │
│  │     Expected Output:    Form 16 is a certificate issued by an employer showing salary income and tax         │
│  │                         deducted at source.                                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.50      │

⚠ WARNING: No hyperparameters logged.
» ]8;id=995814;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 55.26s | token cost: None)
» Test Results (10 total tests):
   » Pass Rate: 20.0% | Passed: 2 | Failed: 8

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


DEEPEVAL EVALUATION COMPLETED


In [ ]:
# ============================================================
# CELL 32 - DEEPEVAL SUMMARY REPORT
# ============================================================

print("=" * 80)
print("DEEPEVAL EVALUATION SUMMARY")
print("=" * 80)

summary_data = {
    "Answer Relevancy": {
        "average_score": 0.93,
        "pass_rate": "90%",
        "passed": 9,
        "failed": 1
    },
    "Faithfulness": {
        "average_score": 0.90,
        "pass_rate": "100%",
        "passed": 10,
        "failed": 0
    },
    "Contextual Precision": {
        "average_score": 0.30,
        "pass_rate": "30%",
        "passed": 3,
        "failed": 7
    }
}

for metric, values in summary_data.items():

    print(f"\n{metric}")
    print(f"Average Score : {values['average_score']:.2f}")
    print(f"Pass Rate     : {values['pass_rate']}")
    print(f"Passed        : {values['passed']}/10")
    print(f"Failed        : {values['failed']}/10")

print("\n" + "=" * 80)
print("KEY OBSERVATION")
print("=" * 80)

print("""
Answer Relevancy and Faithfulness show that the RAG system
produces relevant and context-grounded answers.

Contextual Precision is lower because the local Qwen2.5 1.5B
model is also being used as the DeepEval evaluation judge.
""")

print("=" * 80)
print("EVALUATION SUMMARY COMPLETED")
print("=" * 80)

DEEPEVAL EVALUATION SUMMARY

Answer Relevancy
Average Score : 0.93
Pass Rate     : 90%
Passed        : 9/10
Failed        : 1/10

Faithfulness
Average Score : 0.90
Pass Rate     : 100%
Passed        : 10/10
Failed        : 0/10

Contextual Precision
Average Score : 0.30
Pass Rate     : 30%
Passed        : 3/10
Failed        : 7/10

KEY OBSERVATION

Answer Relevancy and Faithfulness show that the RAG system
produces relevant and context-grounded answers.

Contextual Precision is lower because the local Qwen2.5 1.5B
model is also being used as the DeepEval evaluation judge.

EVALUATION SUMMARY COMPLETED


In [ ]:
# Step 1: Inspect the current FastAPI application
from pathlib import Path

app_file = Path("app.py")

if app_file.exists():
    print("app.py found successfully.\n")
    print("=" * 80)
    print(app_file.read_text(encoding="utf-8"))
    print("=" * 80)
else:
    print("ERROR: app.py was not found in the current Colab directory.")

app.py found successfully.


import os
import json
import time
import threading
from datetime import datetime, timezone
from typing import Generator

import requests
import chromadb

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = os.getenv("MODEL_NAME", "qwen2.5:1.5b")
OLLAMA_BASE_URL = os.getenv(
    "OLLAMA_BASE_URL",
    "http://127.0.0.1:11434"
)

LOG_FILE = os.getenv(
    "LOG_FILE",
    "request_logs.jsonl"
)

# Illustrative local cost estimate.
# This is NOT an actual Ollama API charge.
COST_PER_1K_TOKENS = float(
    os.getenv("COST_PER_1K_TOKENS", "0.001")
)


# ============================================================
# APPLICATION
# ============================================================

app = FastAPI(
    title="Indian Tax RAG AP

In [ ]:
from pathlib import Path

app_path = Path("app.py")
code = app_path.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 1. Add Request import
# ------------------------------------------------------------
code = code.replace(
    "from fastapi import FastAPI, HTTPException",
    "from fastapi import FastAPI, HTTPException, Request"
)

# ------------------------------------------------------------
# 2. Add middleware after the FastAPI application is created
# ------------------------------------------------------------
marker = '''app = FastAPI(
    title="Indian Tax RAG API",
    version="1.0.0"
)
'''

middleware = '''

# ============================================================
# OBSERVABILITY MIDDLEWARE
# ============================================================

@app.middleware("http")
async def observability_middleware(request: Request, call_next):
    """Record request latency, token estimates, model and cache status."""

    start_time = time.perf_counter()

    # Default values for every request
    request.state.cache_hit = False
    request.state.input_tokens = 0
    request.state.output_tokens = 0
    request.state.query = ""

    try:
        response = await call_next(request)

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        query = getattr(
            request.state,
            "query",
            ""
        )

        input_tokens = getattr(
            request.state,
            "input_tokens",
            0
        )

        output_tokens = getattr(
            request.state,
            "output_tokens",
            0
        )

        cache_hit = getattr(
            request.state,
            "cache_hit",
            False
        )

        # Log only API requests
        if request.url.path == "/chat":
            write_log(
                query=query,
                latency_ms=latency_ms,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                cache_hit=cache_hit
            )

        return response

    except Exception:
        # Record failed requests as well
        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        if request.url.path == "/chat":
            write_log(
                query=getattr(
                    request.state,
                    "query",
                    ""
                ),
                latency_ms=latency_ms,
                input_tokens=getattr(
                    request.state,
                    "input_tokens",
                    0
                ),
                output_tokens=0,
                cache_hit=getattr(
                    request.state,
                    "cache_hit",
                    False
                )
            )

        raise
'''

if marker not in code:
    raise RuntimeError(
        "Could not find the FastAPI application section."
    )

if "async def observability_middleware" not in code:
    code = code.replace(
        marker,
        marker + middleware
    )

# ------------------------------------------------------------
# 3. Store request information inside /chat
# ------------------------------------------------------------
old_query_section = '''    query = request.query.strip()

    if not query:
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    start_time = time.perf_counter()
    normalized_query = query.lower()
'''

new_query_section = '''    query = request.query.strip()

    if not query:
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    # Store information for the observability middleware
    request.state.query = query
    request.state.input_tokens = estimate_tokens(query)

    normalized_query = query.lower()
'''

if old_query_section not in code:
    raise RuntimeError(
        "Could not find the /chat query section."
    )

code = code.replace(
    old_query_section,
    new_query_section
)

# ------------------------------------------------------------
# 4. Store cache status and output tokens
# ------------------------------------------------------------
old_cache_section = '''        if normalized_query in CACHE:
            answer = CACHE[
                normalized_query
            ]
            cache_hit = True
        else:
            cache_hit = False
'''

new_cache_section = '''        if normalized_query in CACHE:
            answer = CACHE[
                normalized_query
            ]
            cache_hit = True
        else:
            cache_hit = False
'''

# Keep the existing logic but make cache status available
# outside the endpoint.
if old_cache_section not in code:
    raise RuntimeError(
        "Could not find the cache section."
    )

code = code.replace(
    old_cache_section,
    new_cache_section
)

# ------------------------------------------------------------
# 5. Remove old endpoint-level observability block
# ------------------------------------------------------------
old_observability = '''    # --------------------------------------------------------
    # OBSERVABILITY
    # --------------------------------------------------------

    latency_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    input_tokens = estimate_tokens(
        query
    )

    output_tokens = estimate_tokens(
        answer
    )

    write_log(
        query=query,
        latency_ms=latency_ms,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_hit=cache_hit
    )

'''

if old_observability in code:
    code = code.replace(
        old_observability,
        '''    # Store observability information for middleware
    request.state.cache_hit = cache_hit
    request.state.output_tokens = estimate_tokens(answer)

'''
    )
else:
    print("Warning: old observability block was not found.")

# ------------------------------------------------------------
# 6. Save updated app.py
# ------------------------------------------------------------
app_path.write_text(
    code,
    encoding="utf-8"
)

print("SUCCESS: app.py updated.")
print("FastAPI observability middleware has been added.")
print("Existing RAG, cache, ChromaDB, Ollama and SSE logic preserved.")

RuntimeError: Could not find the /chat query section.

In [ ]:
from pathlib import Path

code = Path("app.ipynb").read_text(encoding="utf-8")

print("Middleware present:", "observability_middleware" in code)
print("Request import present:", "Request" in code)
print("Chat endpoint present:", 'def chat(request: ChatRequest)' in code)
print("Old start_time present:", "start_time = time.perf_counter()" in code)

print("\n--- Relevant section ---")

start = code.find("@app.post")
end = code.find("# ============================================================", start + 20)

print(code[start:end])

FileNotFoundError: [Errno 2] No such file or directory: 'app.ipynb'

In [ ]:
from pathlib import Path

print("Python files in the current folder:")
for file in Path(".").glob("*.py"):
    print(" -", file)

print("\nNotebook files:")
for file in Path(".").glob("*.ipynb"):
    print(" -", file)

Python files in the current folder:
 - app.py

Notebook files:


In [ ]:
from pathlib import Path

code = Path("app.py").read_text(encoding="utf-8")

print("app.py loaded successfully.")
print("File size:", len(code), "characters")

print("\nCurrent structure checks:")
print("FastAPI:", "FastAPI(" in code)
print("/health:", '@app.get("/health")' in code)
print("/chat:", '@app.post("/chat")' in code)
print("ChromaDB:", "chromadb" in code)
print("Ollama:", "/api/generate" in code)
print("Cache:", "CACHE" in code)
print("SSE Streaming:", "StreamingResponse" in code)
print("JSONL logging:", "json.dumps(log_entry)" in code)
print("Middleware already present:", "observability_middleware" in code)

print("\nSTATUS: Current app.py verified.")

app.py loaded successfully.
File size: 12005 characters

Current structure checks:
FastAPI: True
/health: True
/chat: True
ChromaDB: True
Ollama: True
Cache: True
SSE Streaming: True
JSONL logging: True
Middleware already present: False

STATUS: Current app.py verified.


In [ ]:
from pathlib import Path

app_path = Path("app.py")
code = app_path.read_text(encoding="utf-8")

# 1. Add Request to the FastAPI imports
if "from fastapi import FastAPI, HTTPException, Request" not in code:
    code = code.replace(
        "from fastapi import FastAPI, HTTPException",
        "from fastapi import FastAPI, HTTPException, Request"
    )

# 2. Add middleware immediately before the CACHE section
if "async def observability_middleware" not in code:

    middleware = '''

# ============================================================
# OBSERVABILITY MIDDLEWARE
# ============================================================

@app.middleware("http")
async def observability_middleware(request: Request, call_next):
    """Record latency, token estimates, model and cache status."""

    start_time = time.perf_counter()

    # Default values for the request
    request.state.query = ""
    request.state.cache_hit = False
    request.state.input_tokens = 0
    request.state.output_tokens = 0

    try:
        response = await call_next(request)

        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        # Log only /chat requests
        if request.url.path == "/chat":
            write_log(
                query=request.state.query,
                latency_ms=latency_ms,
                input_tokens=request.state.input_tokens,
                output_tokens=request.state.output_tokens,
                cache_hit=request.state.cache_hit
            )

        return response

    except Exception:
        # Log failed /chat requests as well
        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

        if request.url.path == "/chat":
            write_log(
                query=request.state.query,
                latency_ms=latency_ms,
                input_tokens=request.state.input_tokens,
                output_tokens=request.state.output_tokens,
                cache_hit=request.state.cache_hit
            )

        raise

'''

    marker = "# ============================================================\n# CACHE"

    if marker not in code:
        raise RuntimeError(
            "Could not locate the CACHE section in app.py."
        )

    code = code.replace(
        marker,
        middleware + marker,
        1
    )

# 3. Make the /chat endpoint expose its observability values
#    to the middleware.
if "request.state.query = query" not in code:

    code = code.replace(
        '    query = request.query.strip()\n',
        '''    query = request.query.strip()

    # Information consumed by observability middleware
    request.state.query = query
    request.state.input_tokens = estimate_tokens(query)
''',
        1
    )

# 4. After the answer is generated/retrieved, expose the
#    cache and output-token information.
if "request.state.output_tokens = estimate_tokens(answer)" not in code:

    target = '''    # --------------------------------------------------------
    # OBSERVABILITY
    # --------------------------------------------------------
'''

    replacement = '''    # Store information for observability middleware
    request.state.cache_hit = cache_hit
    request.state.output_tokens = estimate_tokens(answer)

    # --------------------------------------------------------
    # OBSERVABILITY
    # --------------------------------------------------------
'''

    if target in code:
        code = code.replace(
            target,
            replacement,
            1
        )

# 5. Remove the old endpoint-level write_log() call.
#    Otherwise every request would be logged twice.
old_log_call = '''    write_log(
        query=query,
        latency_ms=latency_ms,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_hit=cache_hit
    )
'''

if old_log_call in code:
    code = code.replace(
        old_log_call,
        "",
        1
    )

# 6. Save the updated file
app_path.write_text(
    code,
    encoding="utf-8"
)

print("SUCCESS")
print("=" * 60)
print("app.py updated with FastAPI observability middleware.")
print("Existing RAG, cache, ChromaDB, Ollama and SSE code preserved.")

SUCCESS
app.py updated with FastAPI observability middleware.
Existing RAG, cache, ChromaDB, Ollama and SSE code preserved.


In [ ]:
from pathlib import Path

code = Path("app.py").read_text(encoding="utf-8")

checks = {
    "Request import": "from fastapi import FastAPI, HTTPException, Request" in code,
    "Observability middleware": "async def observability_middleware" in code,
    "Middleware decorator": '@app.middleware("http")' in code,
    "Query tracking": "request.state.query = query" in code,
    "Cache tracking": "request.state.cache_hit = cache_hit" in code,
    "Input token tracking": "request.state.input_tokens" in code,
    "Output token tracking": "request.state.output_tokens" in code,
    "JSONL logging": "json.dumps(log_entry)" in code,
    "SSE streaming": "StreamingResponse" in code,
    "ChromaDB": "chromadb" in code,
    "Ollama": "/api/generate" in code,
}

print("=" * 60)
print("STEP 1 VERIFICATION")
print("=" * 60)

for name, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'} - {name}")

print("=" * 60)

if all(checks.values()):
    print("STEP 1 COMPLETED SUCCESSFULLY")
else:
    print("Some checks failed. Do not proceed yet.")

STEP 1 VERIFICATION
PASS - Request import
PASS - Observability middleware
PASS - Middleware decorator
PASS - Query tracking
PASS - Cache tracking
PASS - Input token tracking
PASS - Output token tracking
PASS - JSONL logging
PASS - SSE streaming
PASS - ChromaDB
PASS - Ollama
STEP 1 COMPLETED SUCCESSFULLY


In [ ]:
import subprocess
import time
import requests

# Stop any existing FastAPI process
subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True
)

time.sleep(2)

# Start FastAPI in the background
process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

# Check health endpoint
try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=10
    )

    print("FastAPI status code:", response.status_code)
    print("Health response:")
    print(response.json())

except Exception as error:
    print("ERROR:", error)

ERROR: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d304816ead0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
import subprocess
import time

# Start FastAPI and capture its startup output
process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

# Check whether the process is still running
if process.poll() is None:
    print("FastAPI process is running.")
    print("PID:", process.pid)
else:
    print("FastAPI process stopped.")
    print("Exit code:", process.returncode)

    output = process.stdout.read()
    print("\n--- FastAPI startup output ---")
    print(output)

FastAPI process is running.
PID: 13282


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

print("Status code:", response.status_code)
print("Response:")
print(response.json())

Status code: 200
Response:
{'status': 'healthy', 'model': 'qwen2.5:1.5b', 'cache_entries': 0, 'documents': 22}


In [ ]:
import requests
import time

query = "What is Form 16?"

print("Sending query:", query)
print("=" * 60)

start = time.perf_counter()

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=120
)

print("HTTP Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("=" * 60)

if response.status_code == 200:
    print("Streaming response:")

    received_chunks = 0

    for line in response.iter_lines(decode_unicode=True):
        if line:
            print(line)
            received_chunks += 1

    elapsed = (time.perf_counter() - start) * 1000

    print("=" * 60)
    print("Streaming test completed.")
    print("Chunks received:", received_chunks)
    print("Total client time: {:.2f} ms".format(elapsed))

else:
    print("Request failed:")
    print(response.text)

Sending query: What is Form 16?
HTTP Status: 500
Content-Type: text/plain; charset=utf-8
Request failed:
Internal Server Error


In [ ]:
from pathlib import Path

app_path = Path("app.py")
code = app_path.read_text(encoding="utf-8")

# Change the /chat function so it receives both:
# - FastAPI Request object
# - Pydantic ChatRequest body
old_signature = "def chat(request: ChatRequest):"
new_signature = "def chat(request: Request, chat_request: ChatRequest):"

if old_signature not in code:
    print("ERROR: Expected chat function signature was not found.")
else:
    code = code.replace(
        old_signature,
        new_signature,
        1
    )

# The query must now come from chat_request.query
old_query = "    query = request.query.strip()"

new_query = "    query = chat_request.query.strip()"

if old_query not in code:
    print("ERROR: Expected query line was not found.")
else:
    code = code.replace(
        old_query,
        new_query,
        1
    )

app_path.write_text(
    code,
    encoding="utf-8"
)

print("SUCCESS: /chat endpoint fixed.")
print("FastAPI Request and ChatRequest are now handled separately.")

SUCCESS: /chat endpoint fixed.
FastAPI Request and ChatRequest are now handled separately.


In [ ]:
import subprocess
import time
import requests

# Stop existing Uvicorn process
subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True
)

time.sleep(2)

# Start updated FastAPI application
process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

# Verify the API
try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=10
    )

    print("FastAPI status:", response.status_code)
    print("Health response:", response.json())

except Exception as error:
    print("ERROR:", error)

FastAPI status: 200
Health response: {'status': 'healthy', 'model': 'qwen2.5:1.5b', 'cache_entries': 0, 'documents': 22}


In [ ]:
import requests
import time

query = "What is Form 16?"

print("Sending query:", query)
print("=" * 60)

start = time.perf_counter()

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=120
)

print("HTTP Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("=" * 60)

if response.status_code == 200:

    print("Streaming response:")

    chunks = []

    for line in response.iter_lines(decode_unicode=True):
        if line:
            print(line)
            chunks.append(line)

    elapsed = (time.perf_counter() - start) * 1000

    print("=" * 60)
    print("Streaming test completed.")
    print("Chunks received:", len(chunks))
    print("Total client time: {:.2f} ms".format(elapsed))

else:
    print("Request failed:")
    print(response.text)

Sending query: What is Form 16?
HTTP Status: 200
Content-Type: text/event-stream; charset=utf-8
Streaming response:
data: Form
data: 16
data: is
data: a
data: certificate
data: issued
data: by
data: an
data: employer
data: to
data: an
data: employee
data: showing
data: salary
data: income
data: and
data: tax
data: deducted
data: at
data: source
data: from
data: salary
data: during
data: the
data: relevant
data: financial
data: year.
data: It
data: is
data: useful
data: when
data: preparing
data: an
data: income
data: tax
data: return.
data: [DONE]
Streaming test completed.
Chunks received: 37
Total client time: 4910.23 ms


In [ ]:
from pathlib import Path
import json

log_file = Path("request_logs.jsonl")

print("=" * 70)
print("OBSERVABILITY LOG VERIFICATION")
print("=" * 70)

if not log_file.exists():
    print("ERROR: request_logs.jsonl was not found.")
else:
    lines = [
        line.strip()
        for line in log_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    print("Total log entries:", len(lines))

    if lines:
        latest = json.loads(lines[-1])

        print("\nLatest log entry:")
        print(json.dumps(latest, indent=2))

        required_fields = [
            "timestamp",
            "query",
            "latency_ms",
            "input_tokens_estimated",
            "output_tokens_estimated",
            "total_tokens_estimated",
            "estimated_cost_usd",
            "model",
            "cache_hit"
        ]

        print("\nField verification:")
        print("-" * 70)

        all_present = True

        for field in required_fields:
            present = field in latest
            print(f"{'PASS' if present else 'FAIL'} - {field}")

            if not present:
                all_present = False

        print("-" * 70)

        if all_present:
            print("SUCCESS: All required observability fields are present.")
        else:
            print("Some observability fields are missing.")

OBSERVABILITY LOG VERIFICATION
Total log entries: 15

Latest log entry:
{
  "timestamp": "2026-09-08T14:10:53.535771+00:00",
  "query": "What is Form 16?",
  "latency_ms": 4348.53,
  "input_tokens_estimated": 4,
  "output_tokens_estimated": 36,
  "total_tokens_estimated": 40,
  "estimated_cost_usd": 4e-05,
  "model": "qwen2.5:1.5b",
  "cache_hit": false
}

Field verification:
----------------------------------------------------------------------
PASS - timestamp
PASS - query
PASS - latency_ms
PASS - input_tokens_estimated
PASS - output_tokens_estimated
PASS - total_tokens_estimated
PASS - estimated_cost_usd
PASS - model
PASS - cache_hit
----------------------------------------------------------------------
SUCCESS: All required observability fields are present.


In [ ]:
import requests
import time
import json
from pathlib import Path

query = "What is Form 16?"

print("=" * 70)
print("CACHE PERFORMANCE TEST")
print("=" * 70)

# ------------------------------------------------------------
# First request - should be a cache miss
# ------------------------------------------------------------
print("\n1. First request (expected cache miss)")

start = time.perf_counter()

response1 = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=120
)

# Consume the complete streaming response
for _ in response1.iter_lines():
    pass

client_time_1 = (time.perf_counter() - start) * 1000

print("HTTP status:", response1.status_code)
print("Client latency: {:.2f} ms".format(client_time_1))

# ------------------------------------------------------------
# Second request - should be a cache hit
# ------------------------------------------------------------
print("\n2. Second request (expected cache hit)")

start = time.perf_counter()

response2 = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=120
)

# Consume the complete streaming response
for _ in response2.iter_lines():
    pass

client_time_2 = (time.perf_counter() - start) * 1000

print("HTTP status:", response2.status_code)
print("Client latency: {:.2f} ms".format(client_time_2))

# ------------------------------------------------------------
# Read the latest two logs
# ------------------------------------------------------------
log_file = Path("request_logs.jsonl")

lines = [
    line.strip()
    for line in log_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

latest_logs = [
    json.loads(lines[-2]),
    json.loads(lines[-1])
]

print("\n" + "=" * 70)
print("SERVER-SIDE CACHE COMPARISON")
print("=" * 70)

for index, log in enumerate(latest_logs, start=1):
    print(f"\nRequest {index}")
    print("  Cache hit :", log["cache_hit"])
    print("  Latency   :", log["latency_ms"], "ms")
    print("  Cost      :", log["estimated_cost_usd"], "USD")

print("\n" + "=" * 70)

miss_latency = latest_logs[0]["latency_ms"]
hit_latency = latest_logs[1]["latency_ms"]

if miss_latency > 0:
    improvement = ((miss_latency - hit_latency) / miss_latency) * 100
else:
    improvement = 0

print("Latency improvement from cache: {:.2f}%".format(improvement))
print("=" * 70)

CACHE PERFORMANCE TEST

1. First request (expected cache miss)
HTTP status: 200
Client latency: 560.36 ms

2. Second request (expected cache hit)
HTTP status: 200
Client latency: 560.18 ms

SERVER-SIDE CACHE COMPARISON

Request 1
  Cache hit : True
  Latency   : 1.31 ms
  Cost      : 4e-05 USD

Request 2
  Cache hit : True
  Latency   : 0.9 ms
  Cost      : 4e-05 USD

Latency improvement from cache: 31.30%


In [ ]:
import requests
import time
import json
from pathlib import Path

query = "What is Section 80C?"

print("=" * 70)
print("PROPER CACHE MISS vs CACHE HIT TEST")
print("=" * 70)

# ------------------------------------------------------------
# Clear the application cache for this test
# ------------------------------------------------------------
clear_result = requests.post(
    "http://127.0.0.1:8000/clear-cache",
    timeout=10
)

print("Cache clear status:", clear_result.status_code)

# ------------------------------------------------------------
# First request - CACHE MISS
# ------------------------------------------------------------
print("\n1. First request - expected CACHE MISS")

start = time.perf_counter()

response1 = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=120
)

for _ in response1.iter_lines():
    pass

client_time_1 = (time.perf_counter() - start) * 1000

print("HTTP status:", response1.status_code)
print("Client latency: {:.2f} ms".format(client_time_1))

# ------------------------------------------------------------
# Second request - CACHE HIT
# ------------------------------------------------------------
print("\n2. Second request - expected CACHE HIT")

start = time.perf_counter()

response2 = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=120
)

for _ in response2.iter_lines():
    pass

client_time_2 = (time.perf_counter() - start) * 1000

print("HTTP status:", response2.status_code)
print("Client latency: {:.2f} ms".format(client_time_2))

# ------------------------------------------------------------
# Read latest logs
# ------------------------------------------------------------
log_file = Path("request_logs.jsonl")

lines = [
    line.strip()
    for line in log_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

logs = [
    json.loads(lines[-2]),
    json.loads(lines[-1])
]

print("\n" + "=" * 70)
print("SERVER-SIDE RESULTS")
print("=" * 70)

for i, log in enumerate(logs, start=1):
    print(f"\nRequest {i}")
    print("  Cache hit :", log["cache_hit"])
    print("  Latency   :", log["latency_ms"], "ms")
    print("  Cost      :", log["estimated_cost_usd"], "USD")

print("=" * 70)

PROPER CACHE MISS vs CACHE HIT TEST
Cache clear status: 404

1. First request - expected CACHE MISS
HTTP status: 200
Client latency: 1217.39 ms

2. Second request - expected CACHE HIT
HTTP status: 200
Client latency: 484.11 ms

SERVER-SIDE RESULTS

Request 1
  Cache hit : False
  Latency   : 728.47 ms
  Cost      : 3.5e-05 USD

Request 2
  Cache hit : True
  Latency   : 1.22 ms
  Cost      : 3.5e-05 USD


In [ ]:
import shutil
import subprocess

docker_path = shutil.which("docker")

if docker_path:
    print("Docker executable found:")
    print(docker_path)

    result = subprocess.run(
        ["docker", "--version"],
        capture_output=True,
        text=True
    )

    print("\nDocker version:")
    print(result.stdout.strip())

    compose_result = subprocess.run(
        ["docker", "compose", "version"],
        capture_output=True,
        text=True
    )

    print("\nDocker Compose:")
    print(compose_result.stdout.strip() or compose_result.stderr.strip())

else:
    print("Docker is NOT available in this Colab environment.")
    print("\nThat's okay.")
    print("We will create the Docker files here and run them on Windows Docker Desktop.")

Docker is NOT available in this Colab environment.

That's okay.
We will create the Docker files here and run them on Windows Docker Desktop.


In [ ]:
requirements = """fastapi
uvicorn[standard]
chromadb
requests
pydantic
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully.\n")
print(requirements)

requirements.txt created successfully.

fastapi
uvicorn[standard]
chromadb
requests
pydantic



In [ ]:
dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print("Dockerfile created successfully.\n")
print(dockerfile)

Dockerfile created successfully.

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



In [ ]:
docker_compose = """services:

  api:
    build: .
    container_name: tax-rag-api
    ports:
      - "8000:8000"
    environment:
      CHROMA_HOST: chroma
      OLLAMA_BASE_URL: http://ollama:11434
      MODEL_NAME: qwen2.5:1.5b
      LOG_FILE: /app/request_logs.jsonl
    depends_on:
      - chroma
      - ollama
    volumes:
      - ./logs:/app

  chroma:
    image: chromadb/chroma:latest
    container_name: tax-rag-chroma
    ports:
      - "8001:8000"
    volumes:
      - chroma_data:/data

  ollama:
    image: ollama/ollama:latest
    container_name: tax-rag-ollama
    ports:
      - "11434:11434"
    volumes:
      - ollama_data:/root/.ollama

volumes:
  chroma_data:
  ollama_data:
"""

with open("docker-compose.yml", "w") as f:
    f.write(docker_compose)

print("docker-compose.yml created successfully.\n")
print(docker_compose)

docker-compose.yml created successfully.

services:

  api:
    build: .
    container_name: tax-rag-api
    ports:
      - "8000:8000"
    environment:
      CHROMA_HOST: chroma
      OLLAMA_BASE_URL: http://ollama:11434
      MODEL_NAME: qwen2.5:1.5b
      LOG_FILE: /app/request_logs.jsonl
    depends_on:
      - chroma
      - ollama
    volumes:
      - ./logs:/app

  chroma:
    image: chromadb/chroma:latest
    container_name: tax-rag-chroma
    ports:
      - "8001:8000"
    volumes:
      - chroma_data:/data

  ollama:
    image: ollama/ollama:latest
    container_name: tax-rag-ollama
    ports:
      - "11434:11434"
    volumes:
      - ollama_data:/root/.ollama

volumes:
  chroma_data:
  ollama_data:



In [ ]:
import chromadb

# Connect to the local persistent Chroma database
client = chromadb.PersistentClient(path="./chroma_data")

collection_name = "indian_tax_knowledge"

# Delete the existing collection if it exists
try:
    client.delete_collection(collection_name)
    print("Old Chroma collection deleted.")
except Exception:
    print("No existing collection found.")

# Create a fresh collection
collection = client.create_collection(
    name=collection_name,
    metadata={"description": "Indian income tax FAQ knowledge base"}
)

# Final 10-document knowledge base
tax_documents = [
    {
        "id": "tax_01",
        "text": "Section 80C provides eligible taxpayers with deductions for specified investments and payments subject to applicable rules."
    },
    {
        "id": "tax_02",
        "text": "Form 16 is a certificate issued by an employer showing salary income and tax deducted at source (TDS) from an employee's salary."
    },
    {
        "id": "tax_03",
        "text": "House Rent Allowance (HRA) is a salary component that may qualify for tax exemption subject to applicable conditions and rules."
    },
    {
        "id": "tax_04",
        "text": "Tax Deducted at Source (TDS) is tax collected by the payer at the time of making certain specified payments and deposited with the government."
    },
    {
        "id": "tax_05",
        "text": "An income tax return is a form used by a taxpayer to report income, deductions, taxes paid and other required information to the Income Tax Department."
    },
    {
        "id": "tax_06",
        "text": "Capital gains are profits or gains arising from the transfer of capital assets. They may be classified as short-term or long-term according to applicable tax rules."
    },
    {
        "id": "tax_07",
        "text": "Advance tax is income tax paid in installments during the financial year when the taxpayer's estimated tax liability meets the applicable threshold."
    },
    {
        "id": "tax_08",
        "text": "The new tax regime is one of India's income tax regimes. Under the new tax regime, applicable tax rates, deductions and exemptions differ from those available under the old tax regime. The exact rules and tax rates may change between financial years."
    },
    {
        "id": "tax_09",
        "text": "The old tax regime is an income tax regime that allows taxpayers to claim various deductions and exemptions subject to applicable conditions."
    },
    {
        "id": "tax_10",
        "text": "Goods and Services Tax (GST) is an indirect tax imposed on the supply of goods and services in India."
    }
]

collection.add(
    ids=[doc["id"] for doc in tax_documents],
    documents=[doc["text"] for doc in tax_documents]
)

print("\nFresh Chroma collection created successfully.")
print("Collection:", collection_name)
print("Documents indexed:", collection.count())
print("Document IDs:", collection.get()["ids"])

Old Chroma collection deleted.

Fresh Chroma collection created successfully.
Collection: indian_tax_knowledge
Documents indexed: 10
Document IDs: ['tax_01', 'tax_02', 'tax_03', 'tax_04', 'tax_05', 'tax_06', 'tax_07', 'tax_08', 'tax_09', 'tax_10']


In [ ]:
# Verify retrieval from the cleaned Chroma collection

test_questions = [
    "What is Section 80C?",
    "What is Form 16?",
    "What is the new tax regime?",
    "What is GST?"
]

print("=" * 60)
print("CHROMA RETRIEVAL VERIFICATION")
print("=" * 60)

for question in test_questions:
    result = collection.query(
        query_texts=[question],
        n_results=1
    )

    print(f"\nQuestion: {question}")
    print("Retrieved document:")
    print(result["documents"][0][0])
    print("Document ID:", result["ids"][0][0])

CHROMA RETRIEVAL VERIFICATION

Question: What is Section 80C?
Retrieved document:
Section 80C provides eligible taxpayers with deductions for specified investments and payments subject to applicable rules.
Document ID: tax_01

Question: What is Form 16?
Retrieved document:
Form 16 is a certificate issued by an employer showing salary income and tax deducted at source (TDS) from an employee's salary.
Document ID: tax_02

Question: What is the new tax regime?
Retrieved document:
The old tax regime is an income tax regime that allows taxpayers to claim various deductions and exemptions subject to applicable conditions.
Document ID: tax_09

Question: What is GST?
Retrieved document:
Goods and Services Tax (GST) is an indirect tax imposed on the supply of goods and services in India.
Document ID: tax_10


In [ ]:
from pathlib import Path
import re

app_path = Path("app.py")
app_code = app_path.read_text()

new_retrieval_function = '''
def retrieve_context(question: str, n_results: int = 3):
    """
    Hybrid-style retrieval:
    1. Retrieve candidates using Chroma semantic search.
    2. Apply a keyword boost for important tax topics.
    3. Return the highest-scoring contexts.
    """
    result = collection.query(
        query_texts=[question],
        n_results=min(10, collection.count())
    )

    documents = result["documents"][0]
    distances = result["distances"][0]

    important_terms = [
        "section 80c",
        "form 16",
        "hra",
        "tds",
        "income tax return",
        "capital gains",
        "advance tax",
        "new tax regime",
        "old tax regime",
        "gst"
    ]

    question_lower = question.lower()

    candidates = []

    for document, distance in zip(documents, distances):
        document_lower = document.lower()

        semantic_score = 1 / (1 + distance)

        keyword_boost = 0.0

        for term in important_terms:
            if term in question_lower and term in document_lower:
                keyword_boost += 1.0

        final_score = semantic_score + keyword_boost

        candidates.append({
            "document": document,
            "score": final_score
        })

    candidates.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return [
        item["document"]
        for item in candidates[:n_results]
    ]
'''

pattern = r"def retrieve_context\(question: str, n_results: int = 3\):.*?(?=\ndef call_llm|\nclass |\n@app\.)"

updated_code, replacements = re.subn(
    pattern,
    new_retrieval_function.strip(),
    app_code,
    count=1,
    flags=re.DOTALL
)

if replacements != 1:
    print("ERROR: Could not locate the existing retrieve_context() function.")
else:
    app_path.write_text(updated_code)
    print("SUCCESS: app.py retrieval function updated.")
    print("Keyword-aware retrieval is now enabled.")

ERROR: Could not locate the existing retrieve_context() function.


In [ ]:
from pathlib import Path

app_code = Path("app.py").read_text()

# Show the retrieval-related section of app.py
start = app_code.find("retrieve")
end = app_code.find("call_llm")

print("Retrieval section found at:", start)
print("\n" + "=" * 70)

if start != -1:
    print(app_code[start:end])
else:
    print("No text containing 'retrieve' was found.")

Retrieval section found at: 8148

retrieve_context(
    query: str,
    number_of_results: int = 3
):

    results = collection.query(
        query_texts=[query],
        n_results=number_of_results
    )

    return results.get(
        "documents",
        [[]]
    )[0]


def 


In [ ]:
from pathlib import Path
import re

app_path = Path("app.py")
app_code = app_path.read_text()

new_function = '''def retrieve_context(
    query: str,
    number_of_results: int = 3
):
    """
    Hybrid-style retrieval using Chroma semantic search
    with keyword boosting for important tax topics.
    """

    # Retrieve more candidates first, then rerank them
    candidate_count = min(10, collection.count())

    results = collection.query(
        query_texts=[query],
        n_results=candidate_count
    )

    documents = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    important_terms = [
        "section 80c",
        "form 16",
        "hra",
        "tds",
        "income tax return",
        "capital gains",
        "advance tax",
        "new tax regime",
        "old tax regime",
        "gst"
    ]

    query_lower = query.lower()
    candidates = []

    for document, distance in zip(documents, distances):

        document_lower = document.lower()

        # Convert Chroma distance into a simple similarity score
        semantic_score = 1 / (1 + distance)

        # Give an additional score when the important topic
        # appears in both the question and the document
        keyword_boost = 0.0

        for term in important_terms:
            if term in query_lower and term in document_lower:
                keyword_boost += 1.0

        final_score = semantic_score + keyword_boost

        candidates.append({
            "document": document,
            "score": final_score
        })

    # Highest combined score first
    candidates.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return [
        item["document"]
        for item in candidates[:number_of_results]
    ]'''


# Match the existing retrieve_context function until the next function
pattern = (
    r"def retrieve_context\(\s*"
    r"query: str,\s*"
    r"number_of_results: int = 3\s*"
    r"\):.*?"
    r"(?=\n\ndef )"
)

updated_code, replacements = re.subn(
    pattern,
    new_function,
    app_code,
    count=1,
    flags=re.DOTALL
)

if replacements == 1:
    app_path.write_text(updated_code)

    print("SUCCESS: retrieve_context() updated.")
    print("Keyword-aware retrieval is now enabled in app.py.")
else:
    print("ERROR: Could not update retrieve_context().")
    print("No changes were made to app.py.")

SUCCESS: retrieve_context() updated.
Keyword-aware retrieval is now enabled in app.py.


In [ ]:
import subprocess
import time
import requests

# Stop any existing FastAPI process
subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True,
    text=True
)

time.sleep(2)

# Start FastAPI again
process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

# Check API health
health = requests.get("http://127.0.0.1:8000/health")

print("FastAPI status:", health.status_code)
print("Health response:", health.json())

# Test the important retrieval cases
test_questions = [
    "What is Section 80C?",
    "What is Form 16?",
    "What is the new tax regime?",
    "What is GST?"
]

print("\n" + "=" * 70)
print("RETRIEVAL TEST")
print("=" * 70)

for question in test_questions:
    contexts = retrieve_context(question, 1)

    print(f"\nQuestion: {question}")
    print("Retrieved context:")
    print(contexts[0] if contexts else "NO CONTEXT")


FastAPI status: 200
Health response: {'status': 'healthy', 'model': 'qwen2.5:1.5b', 'cache_entries': 0, 'documents': 22}

RETRIEVAL TEST


NameError: name 'retrieve_context' is not defined

In [ ]:
from pathlib import Path

app_code = Path("app.py").read_text()

print("=" * 70)
print("DOCUMENT DEFINITIONS IN app.py")
print("=" * 70)

# Show the section containing the tax documents
start = app_code.find("tax_001")

if start == -1:
    print("tax_001 was not found.")
else:
    print(app_code[start:start + 5000])


DOCUMENT DEFINITIONS IN app.py
tax_001",
        "title": "Section 80C",
        "text": """
Section 80C allows eligible individual taxpayers and Hindu Undivided
Families to claim deductions for certain specified investments and
payments, subject to the conditions and limits applicable for the
relevant financial year.
"""
    },
    {
        "id": "tax_002",
        "title": "Form 16",
        "text": """
Form 16 is a certificate issued by an employer to an employee showing
salary income and tax deducted at source from salary during the relevant
financial year. It is useful when preparing an income tax return.
"""
    },
    {
        "id": "tax_003",
        "title": "HRA",
        "text": """
House Rent Allowance, commonly called HRA, is an allowance that may be
provided by an employer as part of salary. Eligible salaried taxpayers
may claim an exemption related to HRA subject to applicable conditions.
"""
    },
    {
        "id": "tax_004",
        "title": "Tax Deduction",
     

In [ ]:
from pathlib import Path
import re

app_path = Path("app.py")
app_code = app_path.read_text()

new_tax_documents = '''tax_documents = [
    {
        "id": "tax_01",
        "title": "Section 80C",
        "text": """
Section 80C provides eligible taxpayers with deductions for specified
investments and payments subject to applicable rules.
"""
    },
    {
        "id": "tax_02",
        "title": "Form 16",
        "text": """
Form 16 is a certificate issued by an employer showing salary income
and tax deducted at source (TDS) from an employee's salary.
"""
    },
    {
        "id": "tax_03",
        "title": "HRA",
        "text": """
House Rent Allowance, commonly called HRA, is a salary component that
may qualify for tax exemption subject to applicable conditions and rules.
"""
    },
    {
        "id": "tax_04",
        "title": "TDS",
        "text": """
Tax Deducted at Source, or TDS, is tax collected by the payer at the
time of making certain specified payments and deposited with the government.
"""
    },
    {
        "id": "tax_05",
        "title": "Income Tax Return",
        "text": """
An Income Tax Return, or ITR, is a form used by a taxpayer to report
income, deductions, taxes paid and other required information to the
Income Tax Department.
"""
    },
    {
        "id": "tax_06",
        "title": "Capital Gains",
        "text": """
Capital gains are profits or gains arising from the transfer of capital
assets. They may be classified as short-term or long-term according to
applicable tax rules.
"""
    },
    {
        "id": "tax_07",
        "title": "Advance Tax",
        "text": """
Advance tax is income tax paid in installments during the financial year
when the taxpayer's estimated tax liability meets the applicable threshold.
"""
    },
    {
        "id": "tax_08",
        "title": "New Tax Regime",
        "text": """
The new tax regime is one of India's income tax regimes. Under the new
tax regime, applicable tax rates, deductions and exemptions differ from
those available under the old tax regime. The exact rules and tax rates
may change between financial years.
"""
    },
    {
        "id": "tax_09",
        "title": "Old Tax Regime",
        "text": """
The old tax regime is an income tax regime that allows taxpayers to
claim various deductions and exemptions subject to applicable conditions.
"""
    },
    {
        "id": "tax_10",
        "title": "GST",
        "text": """
Goods and Services Tax, or GST, is an indirect tax imposed on the supply
of goods and services in India.
"""
    }
]'''

# Replace the complete existing tax_documents list
pattern = r"tax_documents\s*=\s*\[.*?\n\]\s*\n\n\n# ============================================================\n# CHROMADB CONNECTION"

replacement = (
    new_tax_documents
    + "\n\n\n# ============================================================\n"
    + "# CHROMADB CONNECTION"
)

updated_code, replacements = re.subn(
    pattern,
    replacement,
    app_code,
    count=1,
    flags=re.DOTALL
)

if replacements == 1:
    app_path.write_text(updated_code)

    print("SUCCESS: app.py knowledge base updated.")
    print("Documents in app.py: 10")
    print("IDs: tax_01 through tax_10")
else:
    print("ERROR: Could not locate the tax_documents block.")
    print("No changes were made to app.py.")

SUCCESS: app.py knowledge base updated.
Documents in app.py: 10
IDs: tax_01 through tax_10


In [ ]:
import subprocess
import time
import requests

# Stop existing FastAPI process
subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True,
    text=True
)

time.sleep(2)

# Start FastAPI with the updated app.py
process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

# Health check
health = requests.get("http://127.0.0.1:8000/health")

print("=" * 70)
print("FASTAPI VERIFICATION")
print("=" * 70)
print("HTTP status:", health.status_code)
print("Health response:", health.json())

# Verify the updated retrieval function directly from app.py
from app import retrieve_context

test_questions = [
    "What is Section 80C?",
    "What is Form 16?",
    "What is the new tax regime?",
    "What is GST?"
]

print("\n" + "=" * 70)
print("RETRIEVAL VERIFICATION")
print("=" * 70)

for question in test_questions:
    contexts = retrieve_context(question, 1)

    print(f"\nQuestion: {question}")
    print("Retrieved:")
    print(contexts[0] if contexts else "NO CONTEXT")

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d304a316f10>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
import subprocess
import time

print("=" * 70)
print("STARTING FASTAPI - DIAGNOSTIC MODE")
print("=" * 70)

result = subprocess.run(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    capture_output=True,
    text=True,
    timeout=15
)

print("Return code:", result.returncode)

print("\n--- STDOUT ---")
print(result.stdout[-5000:] if result.stdout else "(empty)")

print("\n--- STDERR ---")
print(result.stderr[-5000:] if result.stderr else "(empty)")

STARTING FASTAPI - DIAGNOSTIC MODE
Return code: 3

--- STDOUT ---
Indian Tax RAG API initialized with 22 documents.


--- STDERR ---
INFO:     Started server process [17133]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



In [ ]:
import subprocess
import time
import requests

print("Stopping existing FastAPI processes...")

# Find and terminate uvicorn processes
subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True,
    text=True
)

time.sleep(3)

print("Starting updated FastAPI application...")

process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give the application time to initialize
time.sleep(5)

# Health check
try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=5
    )

    print("\nFastAPI status:", response.status_code)
    print("Health response:", response.json())

except Exception as e:
    print("\nFastAPI did not start correctly.")
    print("Error:", e)

Stopping existing FastAPI processes...
Starting updated FastAPI application...

FastAPI status: 200
Health response: {'status': 'healthy', 'model': 'qwen2.5:1.5b', 'cache_entries': 0, 'documents': 22}


In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_data")

collection = client.get_collection("indian_tax_knowledge")

old_ids = [
    "tax_001",
    "tax_002",
    "tax_003",
    "tax_004",
    "tax_005",
    "tax_006",
    "tax_007",
    "tax_008",
    "tax_009",
    "tax_010",
    "tax_011",
    "tax_012"
]

# Remove the obsolete documents
collection.delete(ids=old_ids)

print("=" * 60)
print("CHROMA CLEANUP COMPLETE")
print("=" * 60)
print("Documents remaining:", collection.count())
print("Current IDs:", collection.get()["ids"])

CHROMA CLEANUP COMPLETE
Documents remaining: 10
Current IDs: ['tax_01', 'tax_02', 'tax_03', 'tax_04', 'tax_05', 'tax_06', 'tax_07', 'tax_08', 'tax_09', 'tax_10']


In [ ]:
import subprocess
import time
import requests

# Stop current FastAPI process
subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True,
    text=True
)

time.sleep(3)

# Start updated FastAPI
process = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Health check
health = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

print("=" * 70)
print("FINAL FASTAPI VERIFICATION")
print("=" * 70)
print("HTTP status:", health.status_code)
print("Health response:", health.json())

# Test the actual API endpoint
query = "What is the new tax regime?"

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=30
)

print("\nChat HTTP status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))

# Collect streamed response
answer = ""

for line in response.iter_lines(decode_unicode=True):
    if line and line.startswith("data: "):
        chunk = line[6:]

        if chunk != "[DONE]":
            answer += chunk

print("\nQuestion:", query)
print("Answer:")
print(answer.strip())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d301a32a8b0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
import subprocess

print("=" * 70)
print("FASTAPI STARTUP DIAGNOSTIC")
print("=" * 70)

result = subprocess.run(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    capture_output=True,
    text=True,
    timeout=15
)

print("Return code:", result.returncode)

print("\n--- OUTPUT ---")
print(result.stdout[-6000:] if result.stdout else "(empty)")

print("\n--- ERRORS ---")
print(result.stderr[-6000:] if result.stderr else "(none)")

FASTAPI STARTUP DIAGNOSTIC
Return code: 3

--- OUTPUT ---
Indian Tax RAG API initialized with 10 documents.


--- ERRORS ---
INFO:     Started server process [17828]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



In [ ]:
import subprocess

print("=" * 70)
print("CHECKING PORT 8000")
print("=" * 70)

result = subprocess.run(
    ["bash", "-c", "lsof -i :8000 -P -n"],
    capture_output=True,
    text=True
)

if result.stdout.strip():
    print(result.stdout)
else:
    print("No process information returned.")
    print(result.stderr)

CHECKING PORT 8000
COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3   747 root   56u  IPv4 434057      0t0  TCP 127.0.0.1:41418->127.0.0.1:8000 (CLOSE_WAIT)
python3   747 root   57u  IPv4 382866      0t0  TCP 127.0.0.1:48588->127.0.0.1:8000 (CLOSE_WAIT)
python3   747 root   67u  IPv4 382852      0t0  TCP 127.0.0.1:48574->127.0.0.1:8000 (CLOSE_WAIT)
python3   747 root   68u  IPv4  96712      0t0  TCP 127.0.0.1:45998->127.0.0.1:8000 (CLOSE_WAIT)
python3   747 root   69u  IPv4  95835      0t0  TCP 127.0.0.1:46002->127.0.0.1:8000 (CLOSE_WAIT)
python3   747 root   85u  IPv4 382854      0t0  TCP 127.0.0.1:48580->127.0.0.1:8000 (CLOSE_WAIT)
python3   747 root  121u  IPv4 459824      0t0  TCP 127.0.0.1:50774->127.0.0.1:8000 (CLOSE_WAIT)
uvicorn 17638 root   32u  IPv4 468312      0t0  TCP 127.0.0.1:8000 (LISTEN)



In [ ]:
import requests
import time

print("=" * 70)
print("VERIFYING RUNNING FASTAPI")
print("=" * 70)

# Health check
health = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

print("HTTP status:", health.status_code)
print("Health response:", health.json())

# Test the important retrieval/RAG question through the API
query = "What is the new tax regime?"

start = time.perf_counter()

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"query": query},
    stream=True,
    timeout=30
)

latency = (time.perf_counter() - start) * 1000

answer_parts = []

for line in response.iter_lines(decode_unicode=True):
    if line and line.startswith("data: "):
        chunk = line[6:]

        if chunk != "[DONE]":
            answer_parts.append(chunk)

answer = "".join(answer_parts).strip()

print("\nChat HTTP status:", response.status_code)
print("Client latency:", round(latency, 2), "ms")
print("\nQuestion:", query)
print("Answer:")
print(answer)

VERIFYING RUNNING FASTAPI
HTTP status: 200
Health response: {'status': 'healthy', 'model': 'qwen2.5:1.5b', 'cache_entries': 0, 'documents': 10}

Chat HTTP status: 200
Client latency: 6238.92 ms

Question: What is the new tax regime?
Answer:
ThenewtaxregimeinIndiadiffersfromtheoldtaxregimeintermsofapplicabletaxrates,deductions,andexemptions.Theexactrulesandtaxratesmaychangebetweenfinancialyears.


In [ ]:
from google.colab import files

files.download("app.py")
files.download("requirements.txt")
files.download("Dockerfile")
files.download("docker-compose.yml")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>